# LTX-Video 13B — Free GPU Video Generation Pipeline
### 15-second native • 30s+ chunked • Kaggle T4×2 • NF4 Quantized • Gradio UI

> **What this does:** Runs the full LTX-Video 13B distilled model on free Kaggle T4 GPUs (2×15GB)  
> using NF4 quantization, FFN chunking, and dual-GPU memory management.  
> Generates up to 15 seconds of continuous 480p 30fps video natively, or 30+ seconds via autoregressive chunking.  
> Includes duration-aware NIM prompt enhancement and LLM-generated negative prompts.
>
> **License:** Apache 2.0 (same as LTX-Video)  
> **Author:** [Your Name] — [github.com/yourname](https://github.com/yourname)  
> **Model:** [Lightricks/LTX-Video-0.9.8-13B-distilled](https://huggingface.co/Lightricks/LTX-Video-0.9.8-13B-distilled)

## Step 1 — Configuration
Set your Kaggle secrets, output paths, and generation parameters.  
All values below are tuned for Kaggle T4×2 (2×15GB VRAM).

In [34]:
# ═══ Configuration ═══
# Tuned for Kaggle T4×2 (2×15GB VRAM)
import os, sys
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"  # seed reproducibility

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Hardware ─────────────────────────────────────────────────
import torch
n_gpus = torch.cuda.device_count()
total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory / 1e9 for i in range(n_gpus))
TIER = torch.cuda.get_device_name(0).split("(")[0].strip() if n_gpus > 0 else "CPU"
GPU_GEN = 0
GPU_HOLD = 1 if n_gpus > 1 else 0
def vram_used(i=0): return torch.cuda.memory_allocated(i) / 1e9

# ── Secrets ──────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
_sec = UserSecretsClient()
def _get(k):
    try: return _sec.get_secret(k)
    except: return ""
HF_TOKEN = _get("HF_TOKEN")
NIM_API_KEY = _get("NIM_API_KEY")
NIM_API_BASE = "https://integrate.api.nvidia.com/v1"
NIM_MODEL = "meta/llama-3.3-70b-instruct"

# ── Model ────────────────────────────────────────────────────
HF_REPO = "Lightricks/LTX-Video-0.9.8-13B-distilled"
HF_REPO_2B = "Lightricks/LTX-Video-0.9.7-distilled"
UPSAMPLE_REPO = "Lightricks/ltxv-spatial-upscaler-0.9.7"
HAS_UPSAMPLE = False

# ── Generation Parameters ────────────────────────────────────
GEN_FPS = 30
DISTILLED_TIMESTEPS = [1000, 993, 987, 981, 975, 909, 725]  # official non-uniform
DISTILLED_STEPS = len(DISTILLED_TIMESTEPS)
DISTILLED_CFG = 1.0       # MUST be 1.0 for guidance-distilled
TONE_MAP_RATIO = 0.6      # recommended for 0.9.8 distilled
DEFAULT_NEG = "worst quality, inconsistent motion, blurry, jittery, distorted, morphing, flickering, text, written text, words, letters, numbers, title, subtitle, caption, credits, opening credits, watermark, logo, brand, trademark, signature, stamp, timestamp, date stamp, OSD, HUD, UI overlay, channel icon, lower third, chyron, banner, news ticker, scoreboard, graphic overlay"
DECODE_TIMESTEP = 0.05
IMAGE_COND_NOISE = 0.025
MAX_PROMPT_TOKENS = 128  # T5 max_model_length=128 for LTX — tokens beyond this are IGNORED

# ── Video Conditioning (proven working) ──────────────────────
COND_TAIL_FRAMES = 33     # 8k+1 aligned, ~1.1s of motion context
JUNCTION_BLEND = 8        # tiny cosine blend at concat point

# ── Quality-First Presets (research-backed sweet spots) ──────
# Only 480p and 720p — proven to produce highest consistency
# Max durations limited to ≤5 chunks for minimal drift
QUALITY_PRESETS = {
    "720p":  (1280, 704),  # official default resolution
    "480p":  (864, 480),   # best quality-to-duration ratio
}

# ── Chunk size constants (battle-tested on T4) ───────────────
# These are overridden by model loading cell with VRAM-aware computation.
# Approximate values for reference:
#   720p: ~105f/chunk (3.5s) → max ~15s at 5 chunks
#   480p: ~145f/chunk (4.8s) → max ~20s at 5 chunks
MAX_CHUNKS = 5  # research-backed consistency limit

# ── Output ───────────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── VRAM Budget Constants ────────────────────────────────────
VRAM_SAFETY_MARGIN = 0.85     # leave 15% headroom
V2V_CHUNK_OVERHEAD = 1.3      # video-conditioned needs more VRAM
VRAM_ACTIVATION_COEFF = 2.8e-9  # GB per pixel per frame (WITH FFN chunking)
# Without FFN chunking this was 4.5e-9 — chunking reduces FFN peak by ~8×

# T2V = first chunk (no conditioning). VideoExt = chunk 2+ (tail frames eat VRAM).
CHUNK_FRAMES_T2V = {
    "720p": 193,   # 6.4s — TESTED
    "576p": 257,   # 8.6s (interpolated)
    "480p": 449,   # 15.0s — TESTED
    "360p": 577,   # ~19s (extrapolated)
}
CHUNK_FRAMES_VIDEOEXT = {
    "720p": 161,   # 5.4s — conservative
    "576p": 193,   # 6.4s (interpolated)
    "480p": 297,   # 9.9s — TESTED (449 OOMs)
    "360p": 449,   # ~15s (extrapolated)
}
CHUNK_FRAMES_BY_QUALITY = CHUNK_FRAMES_T2V  # backward compat

# ── FFN Chunking ─────────────────────────────────────────────
FFN_CHUNK_SIZE = 512  # tokens per chunk (256=lowest VRAM, 512=good balance)
# Reduces FFN peak from O(N×4D) to O(chunk_size×4D)
# 22680 tokens / 512 = 44 chunks, each tiny → 0.02GB instead of 0.93GB

# ── Kaggle Cache Paths ───────────────────────────────────────
KAGGLE_DATASET_NAME = "ltxv13b-distilled-cache"
KAGGLE_CACHE_PATH = f"/kaggle/input/datasets/damnyadav/{KAGGLE_DATASET_NAME}"
TMP_MODEL_DIR = "/tmp/ltx13b"
TMP_HF_CACHE = "/tmp/hf_cache"
LTX_2B_CACHE = "/tmp/ltx2b"
USE_LATENT_UPSAMPLE = False
UPSAMPLER_PATH = None
HF_UPSAMPLER = "Lightricks/ltxv-spatial-upscaler-0.9.7"
HF_UPSAMPLER_ALT = "a-r-r-o-w/LTX-0.9.7-Latent-Upsampler"
TMP_UPSAMPLER = "/tmp/ltx_upsampler"
NIM_MODEL_FALLBACK = "meta/llama-3.1-8b-instruct"

print(f"Hardware: {n_gpus}x {TIER} ({total_vram_gb:.0f}GB)")
print(f"Max chunks: {MAX_CHUNKS}")
print(f"  720p: ~129f/chunk (~4.3s native, ~18s at 5 chunks)")
print(f"  480p: ~449f/chunk (~15s NATIVE, ~30s+ at 2 chunks)")
print(f"  FFN chunking: {FFN_CHUNK_SIZE} tokens/chunk → 8× lower peak VRAM")
print(f"  (exact values computed after model loading)")


Hardware: 2x Tesla T4 (31GB)
Max chunks: 5
  720p: ~129f/chunk (~4.3s native, ~18s at 5 chunks)
  480p: ~449f/chunk (~15s NATIVE, ~30s+ at 2 chunks)
  FFN chunking: 512 tokens/chunk → 8× lower peak VRAM
  (exact values computed after model loading)


## Step 2 — Install Dependencies
Installs diffusers, bitsandbytes, imageio, gradio, and openai client.

In [35]:
# ═══ Install Dependencies ═══
import subprocess, importlib

def _install(pkg, pip_name=None):
    try:
        importlib.import_module(pkg.split("[")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               pip_name or pkg])

# Core
_install("diffusers", "diffusers>=0.37.0")
_install("transformers", "transformers>=4.46.0")
_install("accelerate")
_install("bitsandbytes")
_install("sentencepiece")
_install("protobuf")
_install("imageio")
_install("imageio_ffmpeg", "imageio[ffmpeg]")
_install("gradio", "gradio>=5.0")
_install("openai")
_install("peft")
_install("safetensors")

# ESRGAN (optional)
try:
    _install("realesrgan")
    _install("basicsr")
except Exception:
    print("⚠️  ESRGAN not available, Lanczos fallback will be used")

import torch
print(f"✅ Dependencies ready")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   cuda:{i}: {name} ({mem:.1f} GB)")


✅ Dependencies ready
   PyTorch: 2.10.0+cu128
   CUDA: True, GPUs: 2
   cuda:0: Tesla T4 (15.6 GB)
   cuda:1: Tesla T4 (15.6 GB)


## Step 3 — Download & Cache Model
Downloads LTX-Video 13B distilled from HuggingFace and caches to Kaggle dataset for faster restarts.  
First run takes ~10 minutes. Subsequent runs load from cache in ~2 minutes.

In [36]:
# ═══ Download + Kaggle Dataset Cache ═══
# Priority 1: Kaggle dataset (pre-uploaded, instant)
# Priority 2: /tmp cache (from previous download this session)
# Priority 3: HuggingFace download (with T5-XXL reuse from 2B cache)

import os, gc, shutil
from pathlib import Path
from huggingface_hub import snapshot_download

def _is_complete(root):
    """Verify transformer, text_encoder, vae, tokenizer, scheduler present."""
    root = Path(root)
    if not (root / "transformer").is_dir() or not any((root / "transformer").rglob("*.safetensors")):
        return False, "transformer missing"
    if not (root / "text_encoder").is_dir() or not any((root / "text_encoder").rglob("*.safetensors")):
        return False, "text_encoder missing"
    if not (root / "vae").is_dir():
        return False, "vae missing"
    if not (root / "tokenizer").is_dir():
        return False, "tokenizer missing"
    if not (root / "scheduler" / "scheduler_config.json").exists():
        return False, "scheduler missing"
    return True, "ok"

def _find_root(base):
    base = Path(base)
    if _is_complete(base)[0]: return str(base)
    for sub in sorted(base.iterdir()):
        if sub.is_dir() and _is_complete(sub)[0]:
            return str(sub)
    return None

print("=" * 60)
print("  Resolving LTX-Video 13B model source ...")
print("=" * 60)

_fresh_dl   = False
LTX13B_PATH = None

_tmp_free_gb = shutil.disk_usage("/tmp").free / 1e9
_has_2b = (os.path.isdir(f"{LTX_2B_CACHE}/text_encoder") and
           any(Path(f"{LTX_2B_CACHE}/text_encoder").rglob("*.safetensors")))
_needed_gb = 29 if _has_2b else 49

print(f"  /tmp free    : {_tmp_free_gb:.1f} GB")
print(f"  2B T5 cache  : {'✅ found — will reuse!' if _has_2b else '❌ not found'}")
print(f"  Space needed : ~{_needed_gb} GB  {'✅' if _tmp_free_gb > _needed_gb else '⚠️ tight!'}")

# Priority 1: Kaggle dataset cache
if os.path.isdir(KAGGLE_CACHE_PATH):
    found = _find_root(KAGGLE_CACHE_PATH)
    if found:
        LTX13B_PATH = found
        print(f"\n✅ Priority 1: Kaggle dataset → {LTX13B_PATH}")

# Priority 2: /tmp from previous download
if not LTX13B_PATH and _is_complete(TMP_MODEL_DIR)[0]:
    LTX13B_PATH = TMP_MODEL_DIR
    print(f"\n✅ Priority 2: /tmp cache → {LTX13B_PATH}")

# Priority 3: Download from HuggingFace
if not LTX13B_PATH:
    print(f"\n📥 Downloading from {HF_REPO} ...")
    os.makedirs(TMP_MODEL_DIR, exist_ok=True)
    
    _ignore = ["*.md", "*.txt", ".gitattributes", "*.png", "*.jpg"]
    if _has_2b:
        print("  → Reusing T5-XXL from 2B cache (saves ~19 GB)")
        _ignore.append("text_encoder/*")
    
    snapshot_download(
        repo_id=HF_REPO, local_dir=TMP_MODEL_DIR,
        ignore_patterns=_ignore, cache_dir=TMP_HF_CACHE,
        token=HF_TOKEN or None,
    )
    
    if _has_2b:
        te_dst = Path(TMP_MODEL_DIR) / "text_encoder"
        te_src = Path(LTX_2B_CACHE) / "text_encoder"
        if not te_dst.exists():
            os.symlink(str(te_src), str(te_dst))
            print(f"  → Symlinked text_encoder from 2B cache")
    
    if _is_complete(TMP_MODEL_DIR)[0]:
        LTX13B_PATH = TMP_MODEL_DIR
        _fresh_dl = True
        print(f"\n✅ Priority 3: Downloaded → {LTX13B_PATH}")
    else:
        raise RuntimeError(f"Download incomplete: {_is_complete(TMP_MODEL_DIR)[1]}")

# ── Download latent upsampler ────────────────────────────────
UPSAMPLER_PATH = None
if USE_LATENT_UPSAMPLE:
    _up_kaggle = os.path.join(KAGGLE_CACHE_PATH, "latent_upsampler")
    if os.path.isdir(_up_kaggle) and any(Path(_up_kaggle).rglob("*.safetensors")):
        UPSAMPLER_PATH = _up_kaggle
        print(f"  Upsampler: Kaggle cache → {UPSAMPLER_PATH}")
    elif os.path.isdir(TMP_UPSAMPLER) and any(Path(TMP_UPSAMPLER).rglob("*.safetensors")):
        UPSAMPLER_PATH = TMP_UPSAMPLER
        print(f"  Upsampler: /tmp cache → {UPSAMPLER_PATH}")
    else:
        for repo in [HF_UPSAMPLER, HF_UPSAMPLER_ALT]:
            try:
                snapshot_download(repo_id=repo, local_dir=TMP_UPSAMPLER,
                                  ignore_patterns=["*.md","*.txt",".gitattributes"],
                                  cache_dir=TMP_HF_CACHE, token=HF_TOKEN or None)
                UPSAMPLER_PATH = TMP_UPSAMPLER
                print(f"  Upsampler: downloaded from {repo}")
                break
            except Exception as e:
                print(f"  ⚠️ {repo} failed: {e}")

if _fresh_dl:
    print(f"\n📋 To skip downloads next time:")
    print(f"   Upload {TMP_MODEL_DIR} as Kaggle dataset '{KAGGLE_DATASET_NAME}'")

# Cleanup HF cache
if os.path.isdir(TMP_HF_CACHE):
    shutil.rmtree(TMP_HF_CACHE, ignore_errors=True)

gc.collect()
print(f"\n✅ Model path: {LTX13B_PATH}")
if UPSAMPLER_PATH: print(f"   Upsampler: {UPSAMPLER_PATH}")


  Resolving LTX-Video 13B model source ...
  /tmp free    : 1310.1 GB
  2B T5 cache  : ❌ not found
  Space needed : ~49 GB  ✅

✅ Priority 1: Kaggle dataset → /kaggle/input/datasets/damnyadav/ltxv13b-distilled-cache

✅ Model path: /kaggle/input/datasets/damnyadav/ltxv13b-distilled-cache


## Step 4 — Load Model (NF4 Quantized, Dual GPU)
Loads transformer in NF4 on cuda:0, T5-XXL encoder on cuda:1, VAE on cuda:0.  
Applies FFN chunking to all 48 transformer blocks for VRAM reduction.  
Sets deterministic flags for seed reproducibility.

In [37]:
# Deterministic flags for seed reproducibility
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ═══ Model Loading — NF4 on Dual T4 (from local cache) ═══
import gc, os, sys, traceback, torch, importlib, subprocess

MODEL_DTYPE = torch.float16  # T4 = sm_75, no native bf16

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers>=0.33.0", "transformers>=4.48.0",
                "accelerate>=1.2.0", "bitsandbytes"],
               check=True, capture_output=True)
importlib.invalidate_caches()
for _m in list(sys.modules.keys()):
    if any(_m.startswith(p) for p in ["transformers","diffusers","huggingface_hub","accelerate"]):
        sys.modules.pop(_m, None)

from diffusers import LTXConditionPipeline, LTXVideoTransformer3DModel, AutoencoderKLLTXVideo
from diffusers import BitsAndBytesConfig as DiffusersBnBConfig
from diffusers.schedulers import FlowMatchEulerDiscreteScheduler
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from transformers import T5EncoderModel, T5TokenizerFast
from transformers import BitsAndBytesConfig as TransformersBnBConfig
print("\u2705 Imports OK")

n_gpus = torch.cuda.device_count()
TIER = "A100" if torch.cuda.get_device_capability(0)[0] >= 8 else "T4"
GPU_GEN  = 0
GPU_HOLD = min(1, n_gpus - 1)
IS_MULTI_GPU = n_gpus > 1
total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory/1e9 for i in range(n_gpus))

# ── CRITICAL: max_memory must list ALL GPUs + cpu for cross-device hooks ──
_margin = "14GiB"
_tight  = "1GiB"
if IS_MULTI_GPU:
    MAX_MEM_TRANSFORMER = {GPU_GEN: _margin, GPU_HOLD: _tight,  "cpu": "8GiB"}
    MAX_MEM_T5          = {GPU_GEN: _tight,  GPU_HOLD: _margin, "cpu": "8GiB"}
else:
    MAX_MEM_TRANSFORMER = {0: _margin, "cpu": "8GiB"}
    MAX_MEM_T5          = {0: _margin, "cpu": "8GiB"}

def vram_free(dev=0):
    f, _ = torch.cuda.mem_get_info(dev)
    return f / 1e9

def vram_used(dev=0):
    f, t = torch.cuda.mem_get_info(dev)
    return (t - f) / 1e9

def get_vram_budget_gen():
    return vram_free(GPU_GEN)

def compute_max_chunk_frames(w, h, vram_budget=None, v2v_mode=False):
    import math
    if vram_budget is None:
        vram_budget = get_vram_budget_gen()
    overhead = V2V_CHUNK_OVERHEAD if v2v_mode else 1.0
    safe = (vram_budget * VRAM_SAFETY_MARGIN) / overhead
    raw = safe / (w * h * VRAM_ACTIVATION_COEFF)
    k = max(1, int((raw - 1) / 8))
    max_nf = 8 * k + 1
    # Clamp to quality table
    px = w * h
    if   px >= 1280*720*0.9:  cap = CHUNK_FRAMES_BY_QUALITY["720p"]
    elif px >= 1024*576*0.9:  cap = CHUNK_FRAMES_BY_QUALITY["576p"]
    elif px >= 864*480*0.9:   cap = CHUNK_FRAMES_BY_QUALITY["480p"]
    else:                     cap = CHUNK_FRAMES_BY_QUALITY["360p"]
    return min(max_nf, cap)

if "pipe" in dir() and pipe is not None:
    print("\u26a1 Pipeline already loaded")
else:
    def _vs():
        return " | ".join(f"cuda:{i} {vram_used(i):.1f}/{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB"
                          for i in range(n_gpus))

    print(f"\n\U0001f527 Loading from {LTX13B_PATH}")
    print(f"   cuda:{GPU_GEN}=Transformer+VAE | cuda:{GPU_HOLD}=T5")
    print(f"   Before: {_vs()}")

    nf4_diff = DiffusersBnBConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=MODEL_DTYPE)
    nf4_t5   = TransformersBnBConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                      bnb_4bit_compute_dtype=MODEL_DTYPE)

    # Patch for NF4 compatibility
    if not getattr(LTXVideoTransformer3DModel, "_no_split_modules", None):
        LTXVideoTransformer3DModel._no_split_modules = []

    print(f"   Loading transformer (NF4) -> cuda:{GPU_GEN} ...")
    transformer = LTXVideoTransformer3DModel.from_pretrained(
        LTX13B_PATH, subfolder="transformer",
        quantization_config=nf4_diff, torch_dtype=MODEL_DTYPE,
        device_map="auto", max_memory=MAX_MEM_TRANSFORMER)
    print(f"   \u2705 Transformer | {_vs()}")

    # ── FFN CHUNKING — reduces peak activation memory by ~8× ──────
    import torch.nn as nn

    class ChunkedFeedForward(nn.Module):
        """Wraps FeedForward to process tokens in chunks along sequence dim.
        Mathematically identical — zero quality loss.
        Reduces FFN peak from O(N×4D) to O(chunk×4D).
        Source: ComfyUI_LTX-2_VRAM_Memory_Management + diffusers PR #10623"""
        def __init__(self, original_ff, chunk_size=512):
            super().__init__()
            self.ff = original_ff
            self.chunk_size = chunk_size
        def forward(self, hidden_states, *args, **kwargs):
            seq_len = hidden_states.shape[1]
            if seq_len <= self.chunk_size:
                return self.ff(hidden_states, *args, **kwargs)
            output = torch.empty_like(hidden_states)
            for start in range(0, seq_len, self.chunk_size):
                end = min(start + self.chunk_size, seq_len)
                output[:, start:end] = self.ff(
                    hidden_states[:, start:end], *args, **kwargs)
            return output

    ffn_count = 0
    for block in transformer.transformer_blocks:
        if hasattr(block, 'ff'):
            block.ff = ChunkedFeedForward(block.ff, chunk_size=FFN_CHUNK_SIZE)
            ffn_count += 1
    print(f"   ✅ FFN chunking: {ffn_count} blocks, chunk_size={FFN_CHUNK_SIZE}")
    print(f"      Peak FFN: ~{FFN_CHUNK_SIZE * 20480 * 2 / 1e9:.3f} GB (was ~0.93 GB)")



    print(f"   Loading T5-XXL (NF4) -> cuda:{GPU_HOLD} ...")
    text_encoder = T5EncoderModel.from_pretrained(
        LTX13B_PATH, subfolder="text_encoder",
        quantization_config=nf4_t5, torch_dtype=MODEL_DTYPE,
        device_map="auto", max_memory=MAX_MEM_T5)
    print(f"   \u2705 T5 | {_vs()}")

    tokenizer = T5TokenizerFast.from_pretrained(LTX13B_PATH, subfolder="tokenizer")

    print(f"   Loading VAE (FP16) -> cuda:{GPU_GEN} ...")
    vae = AutoencoderKLLTXVideo.from_pretrained(
        LTX13B_PATH, subfolder="vae", torch_dtype=MODEL_DTYPE
    ).to(f"cuda:{GPU_GEN}")
    print(f"   \u2705 VAE | {_vs()}")

    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
        LTX13B_PATH, subfolder="scheduler")

    pipe = LTXConditionPipeline(
        transformer=transformer, text_encoder=text_encoder,
        tokenizer=tokenizer, vae=vae, scheduler=scheduler)
    # ── VAE Tiling — aggressive config for low-VRAM decode ──────────
    # The LTX VAE performs the FINAL denoising step during decode.
    # It needs proper timestep conditioning and denormalization.
    # The pipeline handles all of this when output_type="pil".
    # We just need small enough tiles to fit in 3.4GB free VRAM.
    try:
        # Latest diffusers: spatio-temporal tiling (best quality)
        pipe.vae.enable_tiling(
            tile_sample_min_height=128,        # pixels, smaller = less VRAM per tile
            tile_sample_min_width=128,
            tile_sample_min_num_frames=17,      # temporal tiles (~2s chunks)
            tile_sample_stride_height=64,       # 50% overlap for seamless blending
            tile_sample_stride_width=64,
            tile_sample_stride_num_frames=9,    # temporal overlap
        )
        print(f"   ✅ VAE: spatio-temporal tiling (128px, 17f tiles, 50% overlap)")
    except TypeError:
        # Older diffusers: spatial-only tiling
        pipe.vae.enable_tiling(
            tile_sample_min_height=128,
            tile_sample_min_width=128,
            tile_sample_stride_height=64,
            tile_sample_stride_width=64,
        )
        print(f"   ✅ VAE: spatial tiling (128px tiles, 50% overlap)")

    # Latent upsampler
    pipe_upsample = None
    HAS_UPSAMPLE = False
    if USE_LATENT_UPSAMPLE and UPSAMPLER_PATH:
        try:
            from diffusers import LTXLatentUpsamplePipeline
            from diffusers.pipelines.ltx.modeling_latent_upsampler import LTXLatentUpsamplerModel
            up_model = LTXLatentUpsamplerModel.from_pretrained(UPSAMPLER_PATH, torch_dtype=MODEL_DTYPE)
            pipe_upsample = LTXLatentUpsamplePipeline(
                vae=pipe.vae, latent_upsampler=up_model
            ).to(MODEL_DTYPE).to(f"cuda:{GPU_GEN}")
            HAS_UPSAMPLE = True
            print(f"   \u2705 Upsampler | {_vs()}")
        except Exception as e:
            print(f"   \u26a0\ufe0f Upsampler failed: {e}")

    del transformer, text_encoder, tokenizer, vae, scheduler
    gc.collect(); torch.cuda.empty_cache()

    print(f"\n\u2705 Pipeline ready | {_vs()}")
    print(f"   Upsampler: {'\u2705' if HAS_UPSAMPLE else '\u274c'}")
    for qk, (qw, qh) in QUALITY_PRESETS.items():
        mf = compute_max_chunk_frames(qw, qh)
        print(f"   {qk}: max {mf}f/chunk ({mf/GEN_FPS:.1f}s)")


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch cla

✅ Imports OK
⚡ Pipeline already loaded


## Step 4.5 — LoRA Adapter System
Downloads 20 style/camera/effect LoRAs to `/tmp`. Loads on-demand per generation.
**Key constraints:** Max 2 LoRAs (805MB each). VAE offloaded to CPU during denoising to maximize VRAM.
LoRAs are deleted after each generation to restore full VRAM for next run.

In [ ]:
# ═══ LoRA Management — Clean Implementation for NF4 + device_map ═══
#
# KEY INSIGHT: pipe.load_lora_weights() crashes on NF4 because it passes
# _pipeline=self to load_lora_adapter, which triggers enable_sequential_cpu_offload()
# and Params4bit can't handle the re-offloading.
#
# SOLUTION: Call pipe.transformer.load_lora_adapter(state_dict, prefix="transformer")
# directly. With no _pipeline kwarg, _pipeline=None, offloading path is skipped entirely.
# PEFT injects LoRA layers alongside NF4 (QLoRA pattern) — fully supported.
#
# CLEANUP: Use pipe.transformer.delete_adapters(name) directly.
# Never use pipe.unload_lora_weights() — it tries pipeline-level offloading logic.

import os, gc, json as _json, shutil, traceback
from pathlib import Path
from safetensors.torch import load_file, save_file
from huggingface_hub import hf_hub_download

LORA_CACHE_DIR = "/tmp/ltxv_loras"
os.makedirs(LORA_CACHE_DIR, exist_ok=True)

# ── LoRA Registry — verified filenames from huggingface.co/Lightricks/LTXV-LoRAs/tree/main ──
LORA_REGISTRY = [
    {"id":"bullet_time","name":"🎬 Bullet Time","trigger":"bullet-time",
     "cat":"Camera","desc":"Matrix-style slow-mo rotating camera freeze effect",
     "repo":"Lightricks/LTXV-LoRAs","file":"bullet_time_step_02250_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"through_object","name":"🎬 Through Object","trigger":"through-object",
     "cat":"Camera","desc":"Camera passes through solid objects seamlessly",
     "repo":"Lightricks/LTXV-LoRAs","file":"through_object_step_02500_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"snorricam","name":"🎬 Snorricam","trigger":"snorricam",
     "cat":"Camera","desc":"Body-mounted camera POV, subject stays centered",
     "repo":"Lightricks/LTXV-LoRAs","file":"Snorricam_step_02000_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"equirect360","name":"🎬 360° Equirectangular","trigger":"360-equirectangular",
     "cat":"Camera","desc":"Generates 360° panoramic equirectangular video",
     "repo":"Lightricks/LTXV-LoRAs","file":"360-equirectangular_step_09000.safetensors",
     "fmt":"comfyui","strength":0.9},
    {"id":"flying","name":"🎬 Flying","trigger":"flying",
     "cat":"Camera","desc":"Flying/aerial camera movement through scenes",
     "repo":"Lightricks/LTXV-LoRAs","file":"flying_step_02000_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"wallace_gromit","name":"🎨 Wallace & Gromit","trigger":"walgro style",
     "cat":"Style","desc":"Aardman claymation stop-motion aesthetic",
     "repo":"Lightricks/LTXV-LoRAs","file":"walgro_style_step_42000_comfy.safetensors",
     "fmt":"comfyui","strength":0.9},
    {"id":"arcane","name":"🎨 Arcane Style","trigger":"csetiarcane",
     "cat":"Style","desc":"Arcane TV series painterly animation style",
     "repo":"Lightricks/LTXV-LoRAs","file":"arcane_jinx_step_12000_comfy.safetensors",
     "fmt":"comfyui","strength":0.7},
    {"id":"cakeify","name":"✨ Cakeify","trigger":"CAKEIFY",
     "cat":"Effect","desc":"Transforms objects into realistic cake versions",
     "repo":"Lightricks/LTX-Video-Cakeify-LoRA","file":None,
     "fmt":"diffusers","strength":0.8},
    {"id":"melt","name":"✨ Melt","trigger":"M3LTYX",
     "cat":"Effect","desc":"Objects melt and drip downward dramatically",
     "repo":"Lightricks/LTXV-LoRAs","file":"melt_step_02250_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"face_punch","name":"✨ Face Punch","trigger":"Face_punch",
     "cat":"Effect","desc":"Dramatic impact/punch effect on faces",
     "repo":"Lightricks/LTXV-LoRAs","file":"Face_punch_step_02000_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"explosion","name":"✨ Building Explosion","trigger":"Building_explosion",
     "cat":"Effect","desc":"Dramatic building destruction and debris",
     "repo":"Lightricks/LTXV-LoRAs","file":"Building_Explosion_step_02000_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"cargrip","name":"✨ Car Grip","trigger":"CarGrip",
     "cat":"Effect","desc":"Car gripping/drifting visual effect",
     "repo":"Lightricks/LTXV-LoRAs","file":"CarGrip_step_02000_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"deflate","name":"🔄 Deflate","trigger":"DEFLATE",
     "cat":"Transform","desc":"Deflation/shrinking effect on objects",
     "repo":"Lightricks/LTXV-LoRAs","file":"deflate_step_01500_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"unpaint","name":"🔄 Unpaint","trigger":"UNPAINT",
     "cat":"Transform","desc":"Unpainting/dissolving paint-stripping effect",
     "repo":"Lightricks/LTXV-LoRAs","file":"unpaint_step_01500_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"sprout","name":"🔄 Sprout","trigger":"SPROUT",
     "cat":"Transform","desc":"Organic growth and blooming from surfaces",
     "repo":"Lightricks/LTXV-LoRAs","file":"sprout_step_01500_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"tear","name":"🔄 Tear","trigger":"TEAR",
     "cat":"Transform","desc":"Tearing/ripping apart effect on objects",
     "repo":"Lightricks/LTXV-LoRAs","file":"tear_step_01500_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"snek","name":"🐍 Snek","trigger":"SNEK",
     "cat":"Effect","desc":"Snake-like transformation on subjects",
     "repo":"Lightricks/LTXV-LoRAs","file":"snek_step_03750_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"amgery","name":"😤 Amgery","trigger":"AMGERY",
     "cat":"Effect","desc":"Exaggerated angry expression effect",
     "repo":"Lightricks/LTXV-LoRAs","file":"amgery_step_02500_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"sorpresa","name":"😲 Sorpresa","trigger":"SORPRESA",
     "cat":"Effect","desc":"Exaggerated surprise reaction effect",
     "repo":"Lightricks/LTXV-LoRAs","file":"SORPRESA_step_02500_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
    {"id":"fat_elvis","name":"🕺 Fat Elvis","trigger":"FATELVIS",
     "cat":"Style","desc":"Exaggerated Elvis-inspired character style",
     "repo":"Lightricks/LTXV-LoRAs","file":"fatelvis_step_02750_comfy.safetensors",
     "fmt":"comfyui","strength":0.8},
]

LORA_BY_ID = {l["id"]: l for l in LORA_REGISTRY}
# Detailed descriptions for UI display and LLM context
LORA_DETAIL = {
    "bullet_time": {"ui":"Freezes time and orbits the camera 360° around a frozen moment. Best for action scenes, martial arts, sports. Use with fast-action subjects.",
        "llm":"The BULLET TIME LoRA freezes the scene mid-action while the camera orbits around. Write the prompt describing the frozen moment with suspended particles, then describe the camera rotation. Use trigger 'bullet-time' at the start."},
    "through_object": {"ui":"Camera seamlessly passes through walls, glass, or solid objects. Great for transitions, reveals, architectural walkthroughs.",
        "llm":"The THROUGH OBJECT LoRA makes the camera push through a solid surface and emerge on the other side. Describe what's on both sides and the transition. Use trigger 'through-object' at the start."},
    "snorricam": {"ui":"Camera locked to subject's body — subject stays centered while the world moves behind them. Intense, disorienting POV feel.",
        "llm":"The SNORRICAM LoRA locks the camera to the subject's body so they stay perfectly centered while the background moves. Describe the subject walking or moving, emphasize background motion. Use trigger 'snorricam' at the start."},
    "equirect360": {"ui":"Generates 360° equirectangular panoramic video. Best for environments, landscapes, VR-ready content.",
        "llm":"The 360° EQUIRECTANGULAR LoRA generates panoramic video. Describe the full environment surrounding the camera. Use trigger '360-equirectangular' at the start."},
    "flying": {"ui":"Smooth aerial/flying camera movement over landscapes. Great for establishing shots, nature, cityscapes.",
        "llm":"The FLYING LoRA creates smooth aerial footage. Describe landscape from above, mention banking turns, altitude changes, what passes below. Use trigger 'flying' at the start."},
    "wallace_gromit": {"ui":"Aardman-style claymation stop-motion look. Warm, charming, handmade feel. Works with any subject.",
        "llm":"The WALLACE & GROMIT LoRA applies claymation stop-motion aesthetic. Describe subjects as clay figurines with fingerprint textures, warm lighting, miniature sets. Use trigger 'walgro style' at the start."},
    "arcane": {"ui":"Painterly animation style from Arcane TV series. Rich brushstrokes, dramatic lighting, fantasy atmosphere.",
        "llm":"The ARCANE LoRA applies the painterly animation style from the Arcane series. Use rich visual descriptions with dramatic lighting, visible brushstrokes, fantasy/steampunk elements. Use trigger 'csetiarcane' at the start."},
    "cakeify": {"ui":"Transforms any object into a hyper-realistic cake version. Fondant surfaces, candy details. Best 3-8s.",
        "llm":"The CAKEIFY LoRA transforms objects into realistic cakes. Describe the object on a surface, then its transformation — fondant replacing paint, candy replacing details. Use trigger 'CAKEIFY' at the start."},
    "melt": {"ui":"Objects dramatically melt and drip downward. Liquid pooling, surface distortion. Best 3-8s.",
        "llm":"The MELT LoRA makes objects melt. Describe the object, then it beginning to liquify and drip. Mention pooling liquid, surface distortion. Use trigger 'M3LTYX' at the start."},
    "face_punch": {"ui":"Dramatic impact/punch effect on faces. Shockwave distortion. Best 3-5s.",
        "llm":"The FACE PUNCH LoRA creates impact effects. Describe a face receiving a dramatic impact with shockwave distortion. Use trigger 'Face_punch' at the start."},
    "explosion": {"ui":"Dramatic building destruction with debris, dust clouds, shockwaves. Best 3-8s.",
        "llm":"The EXPLOSION LoRA creates building destruction. Describe a building and then its dramatic explosion with debris, dust, shockwaves. Use trigger 'Building_explosion' at the start."},
    "cargrip": {"ui":"Car gripping/drifting effect. Tire smoke, dramatic angles. Best 5-10s.",
        "llm":"The CAR GRIP LoRA creates car drifting effects. Describe a car performing aggressive turns with tire smoke. Use trigger 'CarGrip' at the start."},
    "deflate": {"ui":"Objects deflate and shrink like losing air. Rubbery collapse. Best 5-10s.",
        "llm":"The DEFLATE LoRA makes objects deflate. Describe an inflated object that begins to lose air and collapse. Use trigger 'DEFLATE' at the start."},
    "unpaint": {"ui":"Paint strips away from surfaces, revealing raw material underneath. Best 5-10s.",
        "llm":"The UNPAINT LoRA strips paint from surfaces. Describe a painted surface where color peels and dissolves away revealing raw material. Use trigger 'UNPAINT' at the start."},
    "sprout": {"ui":"Organic growth — plants, vines, flowers bloom rapidly from surfaces. Best 5-10s.",
        "llm":"The SPROUT LoRA creates rapid organic growth. Describe surfaces where plants/vines/flowers rapidly sprout and bloom. Use trigger 'SPROUT' at the start."},
    "tear": {"ui":"Objects rip and tear apart dramatically. Paper-like tearing effect. Best 5-10s.",
        "llm":"The TEAR LoRA rips objects apart. Describe an object being torn or ripped with visible tears spreading. Use trigger 'TEAR' at the start."},
    "snek": {"ui":"Snake-like transformation on subjects. Serpentine movement. Best 3-5s.",
        "llm":"The SNEK LoRA applies snake-like transformations. Describe subjects with serpentine movement or snake characteristics. Use trigger 'SNEK' at the start."},
    "amgery": {"ui":"Exaggerated angry expression. Comedic rage face distortion. Best 3-5s.",
        "llm":"The AMGERY LoRA creates exaggerated angry expressions. Describe a face becoming comically furious. Use trigger 'AMGERY' at the start."},
    "sorpresa": {"ui":"Exaggerated surprise reaction. Wide eyes, dropped jaw. Best 3-5s.",
        "llm":"The SORPRESA LoRA creates exaggerated surprise. Describe a face reacting with extreme shock. Use trigger 'SORPRESA' at the start."},
    "fat_elvis": {"ui":"Exaggerated Elvis-inspired character transformation. Pompadour, jumpsuit. Best 5-10s.",
        "llm":"The FAT ELVIS LoRA applies an Elvis character transformation. Describe subjects with Elvis characteristics. Use trigger 'FATELVIS' at the start."},
}

LORA_CATEGORIES = sorted(set(l["cat"] for l in LORA_REGISTRY))
_active_lora_ids = []

# ── Key Conversion (ComfyUI → diffusers) ──────────────────
def _convert_comfyui_to_diffusers(src_path, dst_path):
    """Convert key prefix: diffusion_model. → transformer."""
    st = load_file(src_path)
    new_st = {k.replace("diffusion_model.", "transformer."): v for k, v in st.items()}
    save_file(new_st, dst_path)
    return dst_path

# ── Download & Cache ──────────────────────────────────────
def get_lora_path(lora_id):
    """Get local cached path for a LoRA. Downloads + converts if needed."""
    entry = LORA_BY_ID.get(lora_id)
    if not entry:
        return None
    cached = os.path.join(LORA_CACHE_DIR, f"{lora_id}.safetensors")
    if os.path.exists(cached) and os.path.getsize(cached) > 1000:
        return cached
    print(f"  📥 Downloading LoRA: {entry['name']}...")
    try:
        if entry["file"] is None:
            # Diffusers-format repo (Cakeify) — download whole thing
            from huggingface_hub import snapshot_download
            tmp = os.path.join(LORA_CACHE_DIR, f"_tmp_{lora_id}")
            snapshot_download(repo_id=entry["repo"], local_dir=tmp,
                ignore_patterns=["*.md","*.txt","*.png","*.jpg",".gitattributes"],
                token=HF_TOKEN or None)
            for f in Path(tmp).rglob("*.safetensors"):
                shutil.copy2(str(f), cached); break
            shutil.rmtree(tmp, ignore_errors=True)
        else:
            dl = hf_hub_download(repo_id=entry["repo"], filename=entry["file"],
                token=HF_TOKEN or None, cache_dir="/tmp/hf_lora_cache")
            if entry["fmt"] == "comfyui":
                print(f"    Converting ComfyUI → diffusers keys...")
                _convert_comfyui_to_diffusers(dl, cached)
            else:
                shutil.copy2(dl, cached)
        if os.path.exists(cached) and os.path.getsize(cached) > 1000:
            print(f"  ✅ {entry['name']} ({os.path.getsize(cached)/1e6:.0f} MB)")
            return cached
    except Exception as e:
        print(f"  ❌ {entry['name']}: {e}")
        if os.path.exists(cached): os.remove(cached)
    return None

# ── Load / Unload LoRAs ────────────────────────────────────
# LIFECYCLE: delete-before-load. Each generation starts clean.
# VAE offloaded to CPU during LoRA + denoising for ~400MB extra headroom.
# Max 2 LoRAs enforced (2 × 805MB = 1.6GB from ~5.0GB free = 3.4GB for generation).

LORA_VRAM_PER_ADAPTER = 805  # MB, measured from safetensors file size
LORA_MAX_COUNT = 1  # Single LoRA only — 805MB each, 3.8GB free is enough for 480p generation

def load_loras(lora_selections):
    """Load selected LoRAs. Deletes any existing adapters first.
    VAE stays on cuda:0 (needed for I2V encode).
    lora_selections: list of (id, strength) tuples, max 2."""
    global _active_lora_ids
    
    # Enforce max 2
    if len(lora_selections) > LORA_MAX_COUNT:
        lora_selections = lora_selections[:LORA_MAX_COUNT]
        print(f"  ⚠️ Capped to {LORA_MAX_COUNT} LoRAs (VRAM limit)")
    
    # 1. Delete any existing adapters
    _delete_all_adapters()
    
    if not lora_selections:
        return True, "No LoRAs selected (base model)"
    
    # 2. Load each selected LoRA
    loaded, failed = [], []
    for lora_id, strength in lora_selections:
        path = get_lora_path(lora_id)
        if not path:
            failed.append(lora_id); continue
        try:
            sd = load_file(path)
            pipe.transformer.load_lora_adapter(
                sd, adapter_name=lora_id, prefix="transformer"
            )
            loaded.append((lora_id, strength))
            del sd; gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print(f"  ❌ {lora_id}: {e}")
            traceback.print_exc()
            failed.append(lora_id)
    
    # 4. Activate loaded adapters
    if loaded:
        ids = [x[0] for x in loaded]
        wts = [x[1] for x in loaded]
        pipe.transformer.set_adapters(ids, wts)
        _active_lora_ids = ids
        names = [LORA_BY_ID.get(i,{}).get("name",i) for i in ids]
        tw = sum(wts)
        vfree = torch.cuda.mem_get_info(GPU_GEN)[0]/1e9
        msg = f"✅ {len(loaded)} LoRA(s): {', '.join(names)} (Σw={tw:.1f}) | {vfree:.1f}GB free"
        if failed: msg += f" | ❌ {', '.join(failed)}"
        print(f"  {msg}")
        return True, msg
    
    return False, f"❌ Failed: {', '.join(failed)}"

def unload_loras():
    """Delete all adapters and free VRAM."""
    global _active_lora_ids
    _delete_all_adapters()
    _active_lora_ids = []
    gc.collect(); torch.cuda.empty_cache()

def _delete_all_adapters():
    """Delete every adapter from transformer. Verified by checking peft_config."""
    names = list(getattr(pipe.transformer, 'peft_config', {}).keys())
    for name in names:
        try:
            pipe.transformer.delete_adapters(name)
        except Exception as e:
            print(f"  ⚠️ delete_adapter({name}): {e}")
    # Verify
    remaining = list(getattr(pipe.transformer, 'peft_config', {}).keys())
    if remaining:
        print(f"  ⚠️ Adapters still present after delete: {remaining}")
    # Reset PEFT flag if all gone
    if not remaining and hasattr(pipe.transformer, '_hf_peft_config_loaded'):
        pipe.transformer._hf_peft_config_loaded = False
    gc.collect(); torch.cuda.empty_cache()

def get_dynamic_max_frames(w, h, is_first_chunk=True):
    """Compute max frames from ACTUAL free VRAM, not hardcoded tables.
    Call AFTER loading LoRAs (when VRAM is reduced)."""
    vfree = torch.cuda.mem_get_info(GPU_GEN)[0] / 1e9
    # Use the pipeline's own compute function if available
    try:
        return compute_max_chunk_frames(w, h, vram_budget=vfree, v2v_mode=not is_first_chunk)
    except Exception:
        # Fallback: simple estimate
        # ~18MB per frame at 480p, ~45MB at 720p (with FFN chunking)
        px = w * h
        mb_per_frame = 45 if px >= 900000 else 18
        max_f = int((vfree * 1000 * VRAM_SAFETY_MARGIN) / mb_per_frame)
        return ltx_align(max(9, max_f))

def get_active_triggers():
    return " ".join(LORA_BY_ID[i]["trigger"] for i in _active_lora_ids if i in LORA_BY_ID)

# ── LLM Suggestion ───────────────────────────────────────


LORA_TEMPLATES = {
    "none": [
        {"title":"🌅 Cinematic Landscape","prompt":"Aerial establishing shot of misty mountain valley at dawn, golden light filtering through fog, smooth drone forward over pine canopy, 50mm lens, shallow depth of field, natural motion blur","dur":10},
        {"title":"👤 Character Portrait","prompt":"Medium close-up of weathered fisherman mending nets on dock, late afternoon golden light, subtle head turn, calloused hands working rope, shallow focus","dur":8},
        {"title":"🌃 Urban Night","prompt":"Steady tracking shot through neon-lit alley at night, rain-slicked pavement reflecting pink and blue, steam from grates, lone figure with umbrella, 35mm anamorphic","dur":10},
        {"title":"🌊 Ocean Waves","prompt":"Low angle turquoise wave curling in slow motion, sunlight piercing the crest, white foam cascading, water droplets suspended, 100mm macro, golden hour","dur":5},
        {"title":"☕ Cozy Interior","prompt":"Static wide of sunlit reading nook, dust motes in warm light, a cat stretches on velvet cushion, curtains sway in breeze, 35mm, soft focus edges","dur":8},
        {"title":"🏙️ City Timelapse","prompt":"Wide shot of city skyline at dusk, clouds streaking across orange sky, building lights flickering on, traffic flowing below, tripod-locked, 24mm","dur":10},
        {"title":"🌸 Macro Nature","prompt":"Extreme close-up of dewdrop on rose petal at sunrise, refracted garden visible inside droplet, petal texture sharp, background bokeh, 100mm macro","dur":5},
    ],
    "bullet_time": [
        {"title":"🥋 Martial Arts Freeze","prompt":"bullet-time A martial artist mid-flying kick in empty dojo, camera orbits 360 around frozen moment, dust particles suspended in light, wooden floor","dur":8},
        {"title":"🌧️ Raindrop Freeze","prompt":"bullet-time A woman pauses mid-step on rain-soaked street, camera orbits as raindrops hang frozen, neon reflections on wet asphalt, 50mm","dur":8},
        {"title":"🏀 Sports Freeze","prompt":"bullet-time Basketball player frozen mid-dunk, camera rotating around suspended athlete, arena lights creating flares, sweat droplets frozen","dur":5},
        {"title":"💃 Dance Freeze","prompt":"bullet-time Dancer frozen mid-leap above stage, camera orbiting slowly, dress fabric suspended in arc, spotlight beams through haze","dur":8},
        {"title":"🖼️ Portrait Orbit (I2V)","prompt":"bullet-time The subject freezes perfectly still, camera begins smooth orbit around the frozen face, background shifts with parallax, dust motes suspended","dur":8},
    ],
    "through_object": [
        {"title":"🧱 Through Brick Wall","prompt":"through-object Camera pushes forward through brick wall into sunlit garden, particles scatter, warm afternoon light, wide angle, seamless transition","dur":5},
        {"title":"🪟 Through Glass","prompt":"through-object Camera moves through frosted window into cozy cafe, condensation parts around lens, warm amber interior, coffee steam rising","dur":5},
        {"title":"🌊 Through Waterfall","prompt":"through-object Camera pushes through cascading waterfall, water droplets scatter, emerging into hidden grotto with emerald pool, dappled light","dur":5},
        {"title":"🚪 Through Door","prompt":"through-object Camera glides through heavy wooden door into candlelit medieval hall, dust motes in torch light, long table ahead","dur":5},
    ],
    "snorricam": [
        {"title":"🚶 Character Walk","prompt":"snorricam Subject walks determinedly through crowded market, face perfectly centered and sharp, world rushes past, shallow depth of field, golden hour","dur":8},
        {"title":"🏃 Running in Rain","prompt":"snorricam Subject runs through heavy rain on empty street, face centered, streetlights streak behind, water splashing, dramatic side lighting","dur":5},
        {"title":"🎤 Stage Walk","prompt":"snorricam Performer strides confidently toward stage, face centered, backstage corridor rushing past, spotlight growing brighter ahead","dur":8},
        {"title":"🖼️ Portrait Walk (I2V)","prompt":"snorricam The subject begins walking forward slowly, face stays perfectly centered, background shifts and moves, shallow depth of field","dur":8},
    ],
    "equirect360": [
        {"title":"🏛️ Ancient Temple","prompt":"360-equirectangular Standing at center of ancient temple courtyard, towering stone columns wrapped in vines, reflecting pool, panorama reveals archways and peaks, golden hour","dur":10},
        {"title":"🏮 Night Market","prompt":"360-equirectangular Center of neon-lit night market, food stalls stretching every direction, steam from woks, red lanterns swaying, rain-slicked pavement","dur":10},
        {"title":"🌲 Forest Clearing","prompt":"360-equirectangular Sunlit forest clearing, tall pines surrounding, dappled light through canopy, wildflowers dotting meadow, morning mist","dur":10},
        {"title":"🏖️ Tropical Beach","prompt":"360-equirectangular Standing on white sand beach, turquoise ocean ahead, palm trees behind, colorful fishing boats left, cliffs right, golden sunset","dur":10},
        {"title":"🏔️ Mountain Summit","prompt":"360-equirectangular Rocky mountain summit, cloud sea below in every direction, snow-capped peaks piercing through, wind-swept, sunrise colors","dur":10},
    ],
    "flying": [
        {"title":"🍂 Autumn Forest","prompt":"flying Soaring over autumn forest, red and gold canopy to horizon, winding river catching sunlight, smooth banking turns, golden hour, 24mm wide","dur":12},
        {"title":"🏖️ Coastal Cliffs","prompt":"flying Sweeping over dramatic coastal cliffs, turquoise waves crashing against white rocks, seabirds riding thermals, camera banks revealing beach","dur":10},
        {"title":"🏙️ City Skyline","prompt":"flying Gliding over modern city skyline at dusk, glass towers reflecting sunset, traffic on highways below, camera slowly descending toward lights","dur":10},
        {"title":"🌾 Wheat Fields","prompt":"flying Low sweep over golden wheat field at sunset, stalks bending in wind, red barn growing closer in distance, warm amber light flooding frame","dur":10},
        {"title":"❄️ Snowy Mountains","prompt":"flying Soaring over snow-covered alpine peaks, frozen lakes reflecting blue sky, pine forests on slopes, crisp winter morning light","dur":12},
    ],
    "wallace_gromit": [
        {"title":"🍞 Kitchen Scene","prompt":"walgro style Cheerful dog at kitchen table with toast, claymation stop-motion, warm lighting, checkered tablecloth, ears perk up as cheese falls","dur":8},
        {"title":"🌻 Garden Party","prompt":"walgro style Two clay figures picnicking in miniature garden, stop-motion jitter, fingerprint textures, butterfly lands on sandwich, warm light","dur":8},
        {"title":"🎄 Workshop","prompt":"walgro style Clay inventor tinkering with gadget in cluttered workshop, gears scattered, warm desk lamp, fingerprint textures on everything","dur":10},
        {"title":"🖼️ Clay Portrait (I2V)","prompt":"walgro style The subject rendered as claymation figure blinks and looks around, fingerprint textures visible, warm overhead lighting, stop-motion jitter","dur":8},
    ],
    "arcane": [
        {"title":"🌆 Rooftop Scene","prompt":"csetiarcane Hooded figure on rooftop overlooking steampunk city at dusk, painterly brushstrokes, dramatic rim lighting, flowing cape, industrial smoke","dur":10},
        {"title":"✨ Magic Workshop","prompt":"csetiarcane Hands crafting glowing runes on wooden table, blue magical particles floating upward, painterly style, candlelight, visible brushstrokes","dur":8},
        {"title":"⚔️ Battle Stance","prompt":"csetiarcane Warrior in rain, painterly brushstroke aesthetic, glowing weapon, dramatic purple and gold lighting, water droplets frozen","dur":8},
        {"title":"🖼️ Arcane Portrait (I2V)","prompt":"csetiarcane The subject rendered in painterly Arcane style gazes forward, visible brushstrokes across skin, dramatic rim lighting in purple and gold","dur":8},
    ],
    "cakeify": [
        {"title":"🚗 Car Transform","prompt":"CAKEIFY Sports car on turntable slowly transforming into realistic cake, fondant replacing paint, candy headlights, transformation ripples, studio light","dur":5},
        {"title":"👟 Shoe Transform","prompt":"CAKEIFY Designer sneaker on white surface transforms into cake, rubber becomes sponge, laces turn to fondant, studio lighting, macro lens","dur":5},
        {"title":"📱 Phone Transform","prompt":"CAKEIFY Smartphone on marble transforms into cake, screen becomes smooth fondant, buttons turn to candy decorations, studio overhead light","dur":5},
        {"title":"🖼️ Face Cakeify (I2V)","prompt":"CAKEIFY The subject's features slowly transform into hyper-realistic cake, skin becoming fondant, hair turning to buttercream, transformation ripples outward","dur":5},
    ],
    "melt": [
        {"title":"🏆 Trophy Melt","prompt":"M3LTYX Golden trophy on marble pedestal melts downward, liquid gold dripping and pooling at base, dramatic side lighting, black background","dur":5},
        {"title":"🍦 Ice Cream Melt","prompt":"M3LTYX Colorful ice cream sundae melting dramatically, swirls of color flowing down, cherry sinking into pool, warm studio light","dur":5},
        {"title":"🗿 Statue Melt","prompt":"M3LTYX Stone bust on pedestal begins melting like wax, features distorting, liquid stone pooling, dramatic museum lighting, slow drip","dur":8},
        {"title":"🖼️ Face Melt (I2V)","prompt":"M3LTYX The subject's face begins melting downward like warm wax, features dripping, colors blending as form distorts, dramatic side lighting","dur":5},
    ],
    "face_punch": [
        {"title":"👊 Right Hook (I2V)","prompt":"Face_punch A sudden invisible impact strikes the face from the right, shockwave rippling across skin, hair flying outward, slow-motion distortion","dur":3},
        {"title":"💥 Front Impact (I2V)","prompt":"Face_punch Direct frontal impact hits the face, shockwave radiating from center, cheeks distorting, dramatic slow motion, dark background","dur":3},
        {"title":"🌪️ Wind Blast (I2V)","prompt":"Face_punch Powerful wind blast hits face from below, skin rippling upward, hair flying back, dramatic lighting, slow motion capture","dur":3},
    ],
    "explosion": [
        {"title":"🏚️ Warehouse Blast","prompt":"Building_explosion Old warehouse wall erupts in massive explosion, debris flying, dust cloud expanding, shockwave rippling through air, golden hour","dur":5},
        {"title":"🏰 Castle Wall","prompt":"Building_explosion Medieval castle wall explodes from catapult impact, stone blocks tumbling, dust and debris cloud, camera shakes, dramatic light","dur":5},
    ],
    "deflate": [
        {"title":"🦆 Rubber Duck","prompt":"DEFLATE Large inflated rubber duck slowly deflates, body compressing with rubbery folds, air escaping visibly, studio lighting, white background","dur":8},
        {"title":"🖼️ Face Deflate (I2V)","prompt":"DEFLATE The subject's face slowly begins deflating inward like losing air, features compressing, cheeks sinking, rubbery quality, studio light","dur":5},
    ],
    "sprout": [
        {"title":"🧱 Wall Growth","prompt":"SPROUT Bare concrete wall begins sprouting tiny green shoots, rapidly growing into vines with flowers, tendrils reaching upward, morning dew, soft light","dur":8},
        {"title":"📚 Book Growth","prompt":"SPROUT Old book lies open, tiny plants sprouting from pages, growing through text, flowers blooming between words, warm desk lamp light","dur":8},
        {"title":"🖼️ Face Sprout (I2V)","prompt":"SPROUT Tiny green shoots begin emerging from the subject's shoulders and hair, rapidly growing into vines with small leaves, soft natural light","dur":8},
    ],
    "tear": [
        {"title":"📸 Photo Tear","prompt":"TEAR Printed photograph tearing down center, rip spreading diagonally, edges curling like paper, bright white light behind, dramatic contrast","dur":5},
        {"title":"🖼️ Portrait Tear (I2V)","prompt":"TEAR A tear forms across the subject's image from one corner, rip spreading slowly, edges curling back, revealing void behind, dramatic contrast","dur":5},
    ],
    "unpaint": [
        {"title":"🎨 Portrait Unpaint","prompt":"UNPAINT Vibrant painted portrait dissolves like wet watercolor washing off, revealing raw pencil sketch underneath, pigment streaming downward","dur":8},
        {"title":"🖼️ Face Unpaint (I2V)","prompt":"UNPAINT Colors of the subject's face dissolve away like wet paint washed off, revealing pencil sketch underneath, pigment streaming down","dur":8},
    ],
    "amgery": [
        {"title":"😡 Rage Build (I2V)","prompt":"AMGERY Expression shifts to exaggerated fury, brows furrowing intensely, jaw clenching, veins on forehead, camera holds steady close-up, dramatic light","dur":3},
        {"title":"🤬 Slow Burn (I2V)","prompt":"AMGERY Calm expression slowly transforms to extreme anger, nostrils flaring, eyes narrowing, fists clenching visible, dramatic lighting shift","dur":5},
    ],
    "sorpresa": [
        {"title":"😱 Shock Reaction (I2V)","prompt":"SORPRESA Eyes widen in extreme shock, jaw drops open, eyebrows shoot upward, camera pushes in slightly, dramatic lighting shift","dur":3},
        {"title":"🫢 Double Take (I2V)","prompt":"SORPRESA Subject glances away then snaps back with exaggerated surprise, eyes going wide, mouth opening, hands rising, dramatic light","dur":5},
    ],
    "snek": [
        {"title":"🐍 Snake Transform (I2V)","prompt":"SNEK Features begin serpentine transformation, skin taking on scaled texture, eyes narrowing to slits, tongue flickering, sinuous movement, green light","dur":3},
        {"title":"🐍 Full Snek (I2V)","prompt":"SNEK Entire demeanor shifts serpentine, subtle scale patterns on skin, head swaying side to side, eyes becoming reptilian, dramatic lighting","dur":5},
    ],
    "fat_elvis": [
        {"title":"🕺 Elvis Transform","prompt":"FATELVIS Subject transforms with exaggerated pompadour growing upward, rhinestone jumpsuit materializing, dramatic spotlight, slight hip swivel","dur":5},
        {"title":"🎤 Stage Elvis","prompt":"FATELVIS Figure transforms into exaggerated Elvis on Vegas stage, rhinestone cape unfurling, spotlight following, crowd silhouettes below","dur":8},
        {"title":"🖼️ Elvis Portrait (I2V)","prompt":"FATELVIS The subject transforms with pompadour growing, sideburns appearing, collar popping, rhinestone sparkles materializing, spotlight above","dur":5},
    ],
    "cargrip": [
        {"title":"🏎️ Drift Scene","prompt":"CarGrip Sports car aggressive drift on wet track, tire smoke billowing, ground-level camera, rubber marks on asphalt, sunset backlight","dur":8},
        {"title":"🛞 Burnout","prompt":"CarGrip Muscle car standing burnout, rear tires spinning and smoking, camera low behind, rubber particles flying, dramatic side lighting","dur":5},
    ],
}

def download_all_loras():
    print(f"\n📥 Downloading {len(LORA_REGISTRY)} LoRAs to {LORA_CACHE_DIR}...")
    ok = sum(1 for e in LORA_REGISTRY if get_lora_path(e["id"]))
    # Clean HF cache
    shutil.rmtree("/tmp/hf_lora_cache", ignore_errors=True)
    free = shutil.disk_usage("/tmp").free / 1e9
    print(f"\n✅ {ok}/{len(LORA_REGISTRY)} ready | /tmp free: {free:.1f}GB")

download_all_loras()
print(f"   NF4 LoRA mode: transformer.load_lora_adapter (pipeline bypass)")


## Step 5 — Prompt Enhancement System
Duration-aware prompt enhancement using NVIDIA NIM (Llama 3.3 70B).  
Follows the official LTX prompting structure from [docs.ltx.video](https://docs.ltx.video/api-documentation/prompting-guide).  
Also generates scene-specific negative prompts to prevent text artifacts and distortion.

In [9]:
# ═══ Prompt Enhancement System — Official LTX + LLM Negatives ═══

import re, json as _json

_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL | re.IGNORECASE)
def strip_thinking(text):
    if not text: return text
    text = _THINK_RE.sub('', text)
    for p in [re.compile(r'<reasoning>.*?</reasoning>', re.DOTALL|re.IGNORECASE),
              re.compile(r'\[thinking\].*?\[/thinking\]', re.DOTALL|re.IGNORECASE)]:
        text = p.sub('', text)
    return re.sub(r'\s{2,}', ' ', re.sub(r'\n+', ' ', text.strip().replace("**","").replace("*",""))).strip()

def count_tokens(text):
    if hasattr(pipe, "tokenizer") and pipe.tokenizer:
        return pipe.tokenizer(text, return_tensors="pt", add_special_tokens=False)["input_ids"].shape[1]
    return len(text) // 4

def truncate_prompt(text, max_tok=MAX_PROMPT_TOKENS):
    if hasattr(pipe, "tokenizer") and pipe.tokenizer:
        enc = pipe.tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_tok)
        return pipe.tokenizer.decode(enc["input_ids"][0], skip_special_tokens=True)
    return text[:max_tok * 4]

USE_NIM = bool(NIM_API_KEY)
ENHANCE_WORD_THRESHOLD = 80

_NIM_SYSTEM = """You are an expert cinematic director rewriting user prompts for the LTX-Video AI model.

YOUR JOB: Take the user's casual description and transform it into a precise, cinematic LTX-Video prompt.
The user may write casually ("make person punch face", "beautiful sunset scene") — you must translate into professional shot language.

FORMAT: One flowing paragraph. Present tense. Start with trigger word (if any) then DIRECTLY with action.
HARD LIMIT: 70 words max. T5 tokenizer truncates at 128 tokens — every word beyond ~70 is WASTED.

TWO MODES:
T2V (text-to-video): Describe the FULL scene — who, what, where, camera, lighting, atmosphere.
I2V (image-to-video): The model ALREADY SEES the image. Describe ONLY what CHANGES:
  - How the subject MOVES (head turns, expression shifts, body motion)
  - How the CAMERA moves (dolly in, orbit, tracking)
  - How EFFECTS manifest (melt, sprout, transform)
  - NEVER describe appearance, clothing, background, colors already in the image
  - Example bad: "A woman in red dress standing on street" ← model already sees this
  - Example good: "The subject begins walking forward, camera tracking alongside, hair swaying in wind"

STRUCTURE (in order):
1. Trigger word (if LoRA active) — first word(s)
2. Subject + action verb
3. Camera + lens ("50mm f/2.8", "35mm wide", "tracking shot")
4. Lighting + atmosphere (fog, dust, golden hour)
5. End: "natural motion blur" (if space permits)

DURATION RULES:
SHORT (≤5s): 25-40 words. ONE action. Tight framing.
MEDIUM (5-10s): 40-60 words. TWO connected actions.
LONG (10-15s): 55-70 words. THREE phases ending with settling ("eases", "holds", "gradually").

NEVER: emotion labels (sad/happy), endpoint verbs (arrives/finishes), bullets, meta-commentary.
ALWAYS: physical cues (shoulders slump, jaw clenches), extendable verbs (drifts/sways/glides).

Output ONLY the rewritten prompt paragraph. No explanations."""

_NEG_SYSTEM = """Given a video scene, output 15-22 negative prompt terms as a comma-separated list.

ALWAYS include ALL of these (non-negotiable):
worst quality, inconsistent motion, blurry, jittery, distorted, morphing, flickering,
text, written text, words, letters, numbers, title, subtitle, caption, credits, opening credits,
watermark, logo, brand, trademark, copyright, signature, stamp,
timestamp, date stamp, OSD, HUD, UI overlay, channel icon, lower third, chyron, banner,
news ticker, scoreboard, graphic overlay

ADD 3-5 scene-specific terms:
- Water/ocean: moire, high-frequency patterns, shimmering
- Human face: facial distortion, deformed features, extra limbs
- Architecture: banding, crawling texture, perspective warp  
- Dark scene: noise, grain, underexposed
- Animation: do NOT add cartoon or anime
- Fast motion: motion smear, ghosting

Output ONLY comma-separated terms."""

def _call_nim(system, user, max_tokens=400, temp=0.6):
    if not USE_NIM: return None
    try:
        from openai import OpenAI
        client = OpenAI(base_url=NIM_API_BASE, api_key=NIM_API_KEY)
        resp = client.chat.completions.create(
            model=NIM_MODEL, messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user}],
            max_tokens=max_tokens, temperature=temp, top_p=0.9)
        return strip_thinking(resp.choices[0].message.content.strip())
    except Exception as e:
        print(f"  NIM error: {e}")
        try:
            from openai import OpenAI
            client = OpenAI(base_url=NIM_API_BASE, api_key=NIM_API_KEY)
            resp = client.chat.completions.create(
                model=NIM_MODEL_FALLBACK, messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}],
                max_tokens=max_tokens, temperature=temp, top_p=0.9)
            return strip_thinking(resp.choices[0].message.content.strip())
        except Exception as e2:
            print(f"  NIM fallback error: {e2}"); return None

def ensure_quality(prompt):
    lower = prompt.lower()
    add = []
    if "mm" not in lower and "lens" not in lower:
        add.append("Shot on 50mm lens at f/2.8, shallow depth of field.")
    if "motion blur" not in lower and "shutter" not in lower:
        add.append("Natural motion blur, 180-degree shutter equivalent.")
    if add: prompt = prompt.rstrip(". ") + ". " + " ".join(add)
    return prompt

def _build_lora_context(lora_ids, is_i2v=False):
    """Build LLM context string for selected LoRA + mode."""
    if not lora_ids: return ""
    parts = []
    for lid in lora_ids:
        entry = LORA_BY_ID.get(lid, {})
        detail = LORA_DETAIL.get(lid, {})
        if detail.get("llm"):
            mode_note = ""
            if is_i2v:
                if lid in ("face_punch","amgery","sorpresa","snek","melt","cakeify","deflate","sprout","tear","unpaint"):
                    mode_note = " FOR I2V: Describe the effect/transformation happening to the subject. Do NOT describe the subject's appearance."
                elif lid in ("bullet_time","snorricam"):
                    mode_note = " FOR I2V: Describe camera movement around the subject. Do NOT describe the subject's appearance."
                else:
                    mode_note = " FOR I2V: Focus on motion and camera. Do NOT describe what is visible in the image."
            parts.append(detail["llm"] + mode_note)
    if not parts: return ""
    return "\n\nACTIVE LORA (use this in the prompt):\n" + "\n".join(f"- {p}" for p in parts)


def enhance_single_prompt(raw_prompt, duration_s, lora_ids=None, is_i2v=False):
    wc = len(raw_prompt.split())
    if wc >= ENHANCE_WORD_THRESHOLD:
        print(f"  Prompt detailed ({wc} words), keeping as-is")
        return ensure_quality(raw_prompt)
    if duration_s <= 5: label = "SHORT (≤5s)"
    elif duration_s <= 10: label = "MEDIUM (5-10s)"
    else: label = "LONG (10-15s)"
    lora_ctx = _build_lora_context(lora_ids, is_i2v=is_i2v) if lora_ids else ""
    mode_str = "I2V (image-to-video — describe ONLY motion/effects, NOT image contents)" if is_i2v else "T2V (text-to-video — describe full scene)"
    system = _NIM_SYSTEM + lora_ctx
    user_msg = f'Mode: {mode_str}. Duration: {duration_s:.0f}s. Follow {label} rules.\nUser intent: "{raw_prompt}"'
    raw = _call_nim(system, user_msg, max_tokens=250, temp=0.5)
    if raw:
        raw = re.sub(r'^(Here\'?s?|Prompt:?|Sure,?|Output:?|Enhanced:?|Here is)[^.]*[.:]\s*', '', raw, flags=re.IGNORECASE).strip('"').strip()
        if len(raw) > 30:
            enhanced = ensure_quality(raw)
            print(f"  NIM+: {len(enhanced.split())} words for {duration_s:.0f}s")
            return enhanced
    return ensure_quality(raw_prompt)


def enhance_negative(neg, positive_prompt=""):
    base = DEFAULT_NEG
    if positive_prompt and USE_NIM:
        scene_neg = _call_nim(_NEG_SYSTEM, f'Scene: "{positive_prompt[:200]}"', max_tokens=150, temp=0.3)
        if scene_neg and len(scene_neg) > 20:
            scene_neg = re.sub(r'^(Here|Terms|Negative|Output)[^,]*[,:]\s*', '', scene_neg, flags=re.IGNORECASE)
            print(f"  NIM neg: {scene_neg[:80]}...")
            return scene_neg.strip().rstrip(",")
    if not neg or not neg.strip(): return base
    ex = {t.strip().lower() for t in neg.split(",")}
    extras = [t for t in base.split(",") if t.strip().lower() not in ex]
    return neg.rstrip(", ") + (", " + ",".join(extras) if extras else "")

_CHUNK_SYSTEM = """You are a cinematographer planning a multi-shot video.
Given idea, chunk count, seconds per chunk, output JSON:
{"scene_dna": "2-3 sentences under 80 words. Shot, camera, lens (50mm f/2.8), lighting, 3 colors, atmosphere.",
 "actions": ["Chunk 0: 1-2 sentences under 35 words.", "Chunk 1: continuation."]}
Rules: scene_dna constant, extendable verbs only, last chunk settles.
Output ONLY valid JSON."""

def plan_video_prompts(raw_prompt, n_chunks, sec_per_chunk, total_dur):
    if n_chunks == 1:
        enhanced = enhance_single_prompt(raw_prompt, total_dur)
        return [truncate_prompt(enhanced)], enhanced, [""], USE_NIM
    raw = _call_nim(_CHUNK_SYSTEM, f'Video: "{raw_prompt}"\n{total_dur:.0f}s, {n_chunks} chunks of ~{sec_per_chunk:.1f}s.', max_tokens=min(600, 150+n_chunks*60))
    scene_dna, actions, ok = None, [], False
    if raw:
        try:
            p = _json.loads(raw)
            scene_dna = p.get("scene_dna",""); actions = p.get("actions",[])
            if scene_dna and actions: ok = True; print(f"  LLM: DNA={len(scene_dna)}c, {len(actions)} actions")
        except: pass
    if not scene_dna: scene_dna = raw_prompt; actions = [""]*n_chunks; print("  Fallback: raw prompt")
    scene_dna = ensure_quality(scene_dna)
    while len(actions) < n_chunks: actions.append(actions[-1] if actions else "")
    actions = actions[:n_chunks]
    return [truncate_prompt((scene_dna.rstrip(". ")+". "+a.strip()) if a.strip() else scene_dna) for a in actions], scene_dna, actions, ok

PROMPT_TEMPLATES = [
    # ═══ Base Model (No LoRA) ═══
    {"cat":"Base","title":"Cinematic Landscape","prompt":"Aerial view of misty mountain valley at dawn, golden light filtering through fog, smooth drone movement over pine canopy, 50mm f/2.8, natural motion blur","dur":10,"lora":"none"},
    {"cat":"Base","title":"Portrait Close-up","prompt":"Medium close-up of weathered fisherman mending nets on dock, late afternoon golden light, subtle head turn, calloused hands working rope, shallow focus, documentary style","dur":8,"lora":"none"},
    {"cat":"Base","title":"Urban Night","prompt":"Tracking shot through neon-lit Tokyo alley at night, rain-slicked pavement reflecting signs, steam from grates, lone figure with umbrella, 35mm anamorphic, film grain","dur":10,"lora":"none"},
    {"cat":"Base","title":"Espresso Pour","prompt":"Rich dark espresso pours into white ceramic cup, crema forming golden swirl, steam rising, shallow depth of field, warm side lighting, 100mm macro","dur":5,"lora":"none"},
    {"cat":"Base","title":"Ocean Waves","prompt":"Slow-motion wave crashing on rocky shore at sunset, water droplets suspended in golden backlight, foam patterns on dark stone, wide angle, mist rising","dur":8,"lora":"none"},
    
    # ═══ 🎬 Camera LoRAs ═══
    {"cat":"Camera","title":"🎬 Bullet Time — Martial Arts","prompt":"bullet-time A martial artist mid-flying kick in empty dojo, camera orbiting 360° around frozen moment, dust particles suspended in light shafts, wooden floor","dur":8,"lora":"bullet_time"},
    {"cat":"Camera","title":"🎬 Bullet Time — Raindrop","prompt":"bullet-time A single raindrop suspended mid-splash on a puddle surface, camera rotating slowly around the frozen crown of water, city lights reflected below","dur":5,"lora":"bullet_time"},
    {"cat":"Camera","title":"🎬 Through Object — Wall","prompt":"through-object Camera pushes forward through brick wall, emerging into sunlit garden, particles crumbling, warm afternoon light on the other side, wide angle","dur":5,"lora":"through_object"},
    {"cat":"Camera","title":"🎬 Through Object — Glass","prompt":"through-object Camera glides through frosted glass window, ice crystals parting, revealing a cozy firelit cabin interior, warm orange glow, shallow focus","dur":5,"lora":"through_object"},
    {"cat":"Camera","title":"🎬 Snorricam — City Walk","prompt":"snorricam Subject walks through busy city intersection, camera locked to body, face centered and sharp, entire cityscape swaying with each step, golden hour","dur":8,"lora":"snorricam"},
    {"cat":"Camera","title":"🎬 360° — Ancient Temple","prompt":"360-equirectangular Standing at center of vast ancient temple courtyard, towering stone columns in every direction, reflecting pool, dust motes in amber light, birds overhead","dur":10,"lora":"equirect360"},
    {"cat":"Camera","title":"🎬 360° — Night Market","prompt":"360-equirectangular Center of neon-lit night market, food stalls glowing in every direction, steam from woks, crowds moving through aisles, red lanterns swaying","dur":10,"lora":"equirect360"},
    {"cat":"Camera","title":"🎬 Flying — Autumn Forest","prompt":"flying Soaring over vast autumn forest, red and gold canopy to the horizon, winding river catching sunlight, smooth banking turns, epic landscape, 24mm wide","dur":12,"lora":"flying"},
    {"cat":"Camera","title":"🎬 Flying — Coastal Cliffs","prompt":"flying Gliding along dramatic coastal cliffs, turquoise waves crashing below, seabirds at eye level, smooth aerial movement, salt mist rising, golden hour","dur":10,"lora":"flying"},
    
    # ═══ 🎨 Style LoRAs ═══
    {"cat":"Style","title":"🎨 Wallace & Gromit — Kitchen","prompt":"walgro style Cheerful dog at kitchen table with toast and tea, claymation stop-motion, warm lighting, checkered tablecloth, ears perk up as cheese falls","dur":8,"lora":"wallace_gromit"},
    {"cat":"Style","title":"🎨 Wallace & Gromit — Park","prompt":"walgro style Two clay figures on park bench feeding pigeons, stop-motion jitter, miniature trees, warm afternoon light, pigeons bobbing comically","dur":8,"lora":"wallace_gromit"},
    {"cat":"Style","title":"🎨 Arcane — Rooftop","prompt":"csetiarcane Hooded figure on rooftop overlooking steampunk city at dusk, painterly brushstrokes, dramatic rim lighting, flowing cape, industrial smoke stacks","dur":10,"lora":"arcane"},
    {"cat":"Style","title":"🎨 Arcane — Alley","prompt":"csetiarcane Rain-soaked alley with neon signs reflected in puddles, painterly animation style, character leaning against wall, dramatic purple and gold lighting","dur":8,"lora":"arcane"},
    
    # ═══ ✨ Effect LoRAs ═══
    {"cat":"Effect","title":"✨ Cakeify — Sports Car","prompt":"CAKEIFY Sports car on turntable slowly transforms into realistic cake, fondant paint job, candy headlights, transformation ripples front to back, studio lighting","dur":5,"lora":"cakeify"},
    {"cat":"Effect","title":"✨ Melt — Trophy","prompt":"M3LTYX Golden trophy on marble pedestal begins melting, liquid gold dripping and pooling at base, dramatic side lighting, shallow depth of field, dark background","dur":5,"lora":"melt"},
    {"cat":"Effect","title":"✨ Face Punch — Impact","prompt":"Face_punch Sudden invisible impact strikes the face from right side, shockwave rippling across skin, hair flying outward, slow-motion distortion, studio lighting","dur":5,"lora":"face_punch"},
    {"cat":"Effect","title":"✨ Explosion — Building","prompt":"Building_explosion Abandoned warehouse wall erupts outward, debris flying past camera, dust cloud expanding, orange fireball, shockwave rippling through air","dur":5,"lora":"explosion"},
    
    # ═══ 🔄 Transform LoRAs ═══
    {"cat":"Transform","title":"🔄 Sprout — Concrete","prompt":"SPROUT Green shoots burst through cracked concrete, rapidly growing into vines with flowers, tendrils reaching toward sunlight, morning dew forming, macro lens","dur":8,"lora":"sprout"},
    {"cat":"Transform","title":"🔄 Deflate — Sculpture","prompt":"DEFLATE Bronze statue slowly deflates like losing air, features compressing, rubbery collapse, studio lighting tracking the deflation, dramatic shadows","dur":8,"lora":"deflate"},
    {"cat":"Transform","title":"🔄 Tear — Portrait","prompt":"TEAR A tear forms across the image, splitting like paper, revealing bright light behind, rip spreading diagonally, edges curling back, dramatic contrast","dur":5,"lora":"tear"},
    {"cat":"Transform","title":"🔄 Unpaint — Face","prompt":"UNPAINT Colors dissolving from the face like wet paint washing off, revealing pencil sketch underneath, pigment streaming downward, brushstrokes disappearing","dur":8,"lora":"unpaint"},
    
    # ═══ 😤 Expression LoRAs (best for I2V portraits) ═══
    {"cat":"Expression","title":"😤 Amgery — Rage","prompt":"AMGERY Expression shifts to exaggerated fury, brows furrowing intensely, jaw clenching, veins on forehead, camera holds steady close-up, dramatic side lighting","dur":5,"lora":"amgery"},
    {"cat":"Expression","title":"😲 Sorpresa — Shock","prompt":"SORPRESA Eyes widen in extreme shock, jaw drops, eyebrows shoot upward, camera pushes in slightly, dramatic lighting shift emphasizing surprise","dur":5,"lora":"sorpresa"},
    {"cat":"Expression","title":"🐍 Snek — Transform","prompt":"SNEK Features begin serpentine transformation, skin taking scaled texture, eyes narrowing, tongue flickering, sinuous neck movement, green-tinted lighting","dur":5,"lora":"snek"},
    {"cat":"Expression","title":"🕺 Fat Elvis — Transform","prompt":"FATELVIS Exaggerated Elvis pompadour grows upward, rhinestone jumpsuit materializing, spotlight from above, slight hip swivel, camera pulls back","dur":8,"lora":"fat_elvis"},
]



print(f"Prompt system | NIM: {'on' if USE_NIM else 'off'} | LLM pos+neg enhancement")


Prompt system | NIM: on | LLM pos+neg enhancement


## Step 6 — Generation Engine
Core video generation with:
- **Official distilled timesteps** from `ltxv-13b-0.9.8-distilled.yaml`
- **T2V/VideoExt frame caps** (449/297 at 480p, 193/161 at 720p)
- **cuda:1 VAE decode** (10GB free — full single-pass, no tiling artifacts)
- **Autoregressive chunking** with cosine junction blending for 30s+ videos

In [10]:
# ═══ Generation Engine — Custom Timesteps + cuda:1 Decode ═══

import gc, math, os, random, time, uuid, inspect
import numpy as np
from PIL import Image
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
import imageio, torch

# Detect supported pipeline parameters once
_PIPE_PARAMS = set(inspect.signature(pipe.__call__).parameters.keys())
_HAS_TONE_MAP = "tone_map_compression_ratio" in _PIPE_PARAMS
_HAS_TIMESTEPS = "timesteps" in _PIPE_PARAMS
print(f"  Pipeline params: tone_map={_HAS_TONE_MAP}, timesteps={_HAS_TIMESTEPS}")

def ltx_align(nf):
    nf = max(9, int(nf))
    return max(1, round((nf-1)/8)) * 8 + 1

def align32(v): return max(32, (int(v)//32)*32)

def get_max_frames(w, h, is_first_chunk=True):
    px = w * h
    caps = CHUNK_FRAMES_T2V if is_first_chunk else CHUNK_FRAMES_VIDEOEXT
    if px >= 1280*704*0.9: return caps.get("720p", 193)
    if px >= 960*544*0.9:  return caps.get("576p", 193)
    if px >= 832*480*0.9:  return caps.get("480p", 449 if is_first_chunk else 297)
    return caps.get("360p", 577 if is_first_chunk else 449)

def blend_junction(prev, new, n=JUNCTION_BLEND):
    if n <= 0 or len(prev) < n or len(new) < n: return prev + new
    base, tail, head, rest = prev[:-n], prev[-n:], new[:n], new[n:]
    bl = list(base)
    for i in range(n):
        w = 0.5*(1.0-math.cos(math.pi*i/max(1,n-1)))
        a, b = np.array(tail[i]).astype(np.float32), np.array(head[i]).astype(np.float32)
        bl.append(Image.fromarray((a*(1-w)+b*w).clip(0,255).astype(np.uint8)))
    bl.extend(rest)
    return bl

def gen_chunk(prompt, neg, w, h, nf, seed, cond_img=None, cond_vid=None, log_cb=None):
    dev = f"cuda:{GPU_GEN}"
    decode_dev = f"cuda:{GPU_HOLD}"
    gen = torch.Generator(device=dev).manual_seed(seed)
    conds = []
    if cond_vid and len(cond_vid) > 0:
        rv = [f.resize((w,h),Image.LANCZOS) if f.size!=(w,h) else f for f in cond_vid]
        conds.append(LTXVideoCondition(video=rv, frame_index=0))
    elif cond_img is not None:
        ci = cond_img.resize((w,h),Image.LANCZOS) if cond_img.size!=(w,h) else cond_img
        conds.append(LTXVideoCondition(image=ci, frame_index=0))

    kw = dict(
        prompt=truncate_prompt(prompt),
        negative_prompt=truncate_prompt(neg) if neg else "",
        width=w, height=h, num_frames=nf,
        guidance_scale=DISTILLED_CFG,
        decode_timestep=DECODE_TIMESTEP,
        decode_noise_scale=IMAGE_COND_NOISE,
        image_cond_noise_scale=IMAGE_COND_NOISE,
        generator=gen, output_type="latent",
    )
    # Use official timesteps if supported, else fall back to num_inference_steps
    if _HAS_TIMESTEPS:
        kw["timesteps"] = DISTILLED_TIMESTEPS
    else:
        kw["num_inference_steps"] = DISTILLED_STEPS
    # Tone mapping if supported
    if _HAS_TONE_MAP:
        kw["tone_map_compression_ratio"] = TONE_MAP_RATIO
    if conds: kw["conditions"] = conds
    gc.collect(); torch.cuda.empty_cache()

    # Gradual OOM retry: 5% steps from 100% down to 70%
    _oom_fracs = [round(1.0 - i*0.05, 2) for i in range(7)]  # [1.0..0.70]
    for att, frac in enumerate(_oom_fracs):
        try:
            anf = ltx_align(max(9, int(nf*frac)))
            if att > 0:
                kw["num_frames"] = anf
                kw["generator"] = torch.Generator(device=dev).manual_seed(seed)
                gc.collect(); torch.cuda.empty_cache()
                if log_cb: log_cb(f"    OOM retry: {nf}f -> {anf}f ({int(frac*100)}%)")
            latents = pipe(**kw).frames
            if log_cb: log_cb(f"    Decoding on cuda:{GPU_HOLD}...")
            lm = pipe.vae.latents_mean.view(1,-1,1,1,1).to(latents.device, latents.dtype)
            ls = pipe.vae.latents_std.view(1,-1,1,1,1).to(latents.device, latents.dtype)
            latents = latents * ls + lm
            pipe.vae.to(decode_dev)
            latents = latents.to(decode_dev, pipe.vae.dtype)
            temb = torch.tensor([DECODE_TIMESTEP], device=decode_dev, dtype=pipe.vae.dtype)
            with torch.no_grad():
                decoded = pipe.vae.decode(latents, temb, return_dict=False)[0]
            decoded = decoded.squeeze(0).permute(1,2,3,0)
            decoded = ((decoded.float()+1.0)/2.0).clamp(0,1)
            decoded = (decoded*255).to(torch.uint8).cpu().numpy()
            frames = [Image.fromarray(decoded[i]) for i in range(decoded.shape[0])]
            del decoded, latents, temb; torch.cuda.empty_cache()
            pipe.vae.to(dev)
            return frames
        except torch.cuda.OutOfMemoryError:
            try: pipe.vae.to(dev)
            except: pass
            torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache()
            if att == len(_oom_fracs)-1: raise RuntimeError(f"OOM at {int(nf*frac)}f ({int(frac*100)}%)")
    return []

def plan_chunks(total_frames, w, h):
    max_t2v = get_max_frames(w, h, True)
    max_ext = get_max_frames(w, h, False)
    total_frames = ltx_align(total_frames)
    if total_frames <= max_t2v:
        return [{"nf": total_frames, "idx":0, "first":True}]
    cs = [{"nf": min(max_t2v, total_frames), "idx":0, "first":True}]
    gen = cs[0]["nf"]; idx = 1
    while gen < total_frames:
        remaining = total_frames - gen + COND_TAIL_FRAMES
        nf = ltx_align(min(max_ext, remaining))
        if nf < 17: nf = 17
        cs.append({"nf":nf, "idx":idx, "first":False, "new":max(1, nf-COND_TAIL_FRAMES)})
        gen += nf - COND_TAIL_FRAMES; idx += 1
        if idx > MAX_CHUNKS + 2: break
    return cs

def generate_video(prompt, neg, total_frames, w, h, seed,
                   cond_img=None, progress_cb=None, log_cb=None):
    """Generate video. Prompt/neg should already be enhanced (no re-enhancement)."""
    t0 = time.time()
    if seed == -1: seed = random.randint(0, 2**32-1)
    total_frames = ltx_align(total_frames); w = align32(w); h = align32(h)
    chunks = plan_chunks(total_frames, w, h); nc = len(chunks); dur = total_frames/GEN_FPS
    def log(m):
        print(m)
        if log_cb: log_cb(m)

    # For single chunk: use prompt directly (already enhanced by UI)
    # For multi-chunk: need to plan chunks via LLM
    if nc == 1:
        cps = [truncate_prompt(prompt)]
        dna, acts, llm_ok = prompt, [""], False
    else:
        if progress_cb: progress_cb(0.01, "Planning chunks...")
        log("Planning chunk prompts...")
        cps, dna, acts, llm_ok = plan_video_prompts(prompt, nc, dur/nc, dur)

    neg_e = neg if neg else DEFAULT_NEG

    log(f"{w}x{h} | {total_frames}f ({dur:.1f}s) | Seed: {seed} | Chunks: {nc}")
    ts_info = f"timesteps={DISTILLED_TIMESTEPS}" if _HAS_TIMESTEPS else f"steps={DISTILLED_STEPS}"
    log(f"{ts_info} | tone_map={'on' if _HAS_TONE_MAP else 'off'}")

    all_f = []
    for ci, ch in enumerate(chunks):
        nf = ch["nf"]; cp = cps[ci] if ci < len(cps) else cps[-1]
        if progress_cb: progress_cb((ci+0.5)/nc*0.9, f"Chunk {ci+1}/{nc}...")
        if ch["first"]:
            log(f"Chunk 1/{nc} ({nf}f) {'I2V' if cond_img else 'T2V'}...")
            fr = gen_chunk(cp, neg_e, w, h, nf, seed, cond_img=cond_img, log_cb=log)
            all_f = fr
        else:
            tail = all_f[-COND_TAIL_FRAMES:]
            log(f"Chunk {ci+1}/{nc} ({nf}f) VideoExt...")
            fr = gen_chunk(cp, neg_e, w, h, nf, seed+ci, cond_vid=tail, log_cb=log)
            if fr: all_f = blend_junction(all_f, fr[COND_TAIL_FRAMES:])
        log(f"  -> {len(all_f)} frames")
        del fr; gc.collect(); torch.cuda.empty_cache()

    # Auto-extend: if OOM reduced frames below 70% of target, add continuation chunks
    if len(all_f) < total_frames * 0.70 and len(all_f) >= 17:
        deficit = total_frames - len(all_f)
        max_ext = get_max_frames(w, h, False)
        ext_idx = len(chunks)
        while deficit > 8 and ext_idx < len(chunks) + 3:
            tail = all_f[-COND_TAIL_FRAMES:]
            ext_nf = ltx_align(min(max_ext, deficit + COND_TAIL_FRAMES))
            if ext_nf < 17: break
            log(f"  Auto-extend {ext_idx+1} ({ext_nf}f) to fill {deficit}f deficit...")
            try:
                fr = gen_chunk(cp, neg_e, w, h, ext_nf, seed+ext_idx+100, cond_vid=tail, log_cb=log)
                if fr:
                    all_f = blend_junction(all_f, fr[COND_TAIL_FRAMES:])
                    deficit = total_frames - len(all_f)
                    log(f"  -> {len(all_f)}f (deficit: {max(0,deficit)}f)")
                    del fr; gc.collect(); torch.cuda.empty_cache()
                else: break
            except Exception as e:
                log(f"  Auto-extend failed: {e}"); break
            ext_idx += 1
    if not all_f: raise RuntimeError("No frames!")
    el = time.time() - t0
    meta = {"width":w, "height":h, "frames":len(all_f), "duration_s":len(all_f)/GEN_FPS,
            "seed":seed, "steps":len(DISTILLED_TIMESTEPS), "chunks":nc, "gen_time_s":el,
            "llm":llm_ok}
    log(f"Done: {len(all_f)}f ({meta['duration_s']:.1f}s) in {el:.0f}s")
    return all_f, meta

def save_video(frames, path=None, fps=GEN_FPS):
    if path is None:
        path = os.path.join(OUTPUT_DIR, f"ltxv_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}.mp4")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    try:
        with imageio.get_writer(path,fps=fps,codec="libx264",output_params=["-crf","18"],macro_block_size=1) as wr:
            for f in frames: wr.append_data(np.array(f))
    except:
        with imageio.get_writer(path,fps=fps) as wr:
            for f in frames: wr.append_data(np.array(f))
    print(f"Saved: {path}")
    return path

print(f"Engine ready")
for qk,(qw,qh) in QUALITY_PRESETS.items():
    t2v = get_max_frames(qw,qh,True); ext = get_max_frames(qw,qh,False)
    print(f"  {qk}: T2V {t2v}f ({t2v/GEN_FPS:.1f}s) | Ext {ext}f ({ext/GEN_FPS:.1f}s)")


  Pipeline params: tone_map=False, timesteps=True
Engine ready
  720p: T2V 193f (6.4s) | Ext 161f (5.4s)
  480p: T2V 449f (15.0s) | Ext 297f (9.9s)


## Step 7 — Gradio Interface
Two-step workflow: **Enhance** prompts first (review/edit), then **Generate**.  
Includes template library, batch generation, and system diagnostics.

In [ ]:
# ═══ Gradio UI — Clean Enhance-First Flow ═══
import gradio as gr
import math, traceback, re as _re, time as _time, random

ASPECT_MAP = {"16:9 Landscape":16/9, "9:16 Portrait":9/16, "4:3 Standard":4/3, "1:1 Square":1.0}

def resolve_dims(q, asp, img=None):
    bw, bh = QUALITY_PRESETS.get(q, (864, 480))
    r = ASPECT_MAP.get(asp, 16/9)
    if img: r = img.size[0] / img.size[1]
    h = int(math.sqrt(bw*bh/r)); w = int(h*r)
    return align32(w), align32(h)

def build_dur_options(q):
    w, h = QUALITY_PRESETS.get(q, (864, 480))
    t2v = get_max_frames(w, h, True)
    ext = get_max_frames(w, h, False)
    new_per = ext - COND_TAIL_FRAMES; opts = []
    for sec in [3,5,8,10,12,15,20,25,30]:
        nf = ltx_align(int(sec * GEN_FPS))
        nc = 1 if nf <= t2v else 1 + max(1, math.ceil((nf - t2v) / new_per))
        if nc > MAX_CHUNKS: break
        opts.append(f"{sec}s ({'native' if nc==1 else f'{nc} chunks'})")
    return opts

def get_stats():
    f0 = torch.cuda.mem_get_info(0)[0]/1e9
    f1 = torch.cuda.mem_get_info(1)[0]/1e9 if torch.cuda.device_count()>1 else 0
    la = f" | LoRA: {_active_lora_ids[0]}" if _active_lora_ids else ""
    return f"gpu0: {f0:.1f}GB | gpu1: {f1:.1f}GB{la}"

# ── LoRA Handlers ──

def on_lora_change(selection):
    if not selection or selection.startswith("None"):
        tpls = LORA_TEMPLATES.get("none", [])
        info = ("### Base Model (No LoRA)\n"
                "Full VRAM available. Max **15s** native at 480p, **6.4s** at 720p.\n\n"
                "**Best for:** Cinematic scenes, landscapes, portraits, any standard video.\n\n"
                "**Tip:** Write your scene naturally. The enhancer adds camera specs and atmosphere.")
        return info, gr.update(choices=["— pick a template —"]+[t["title"] for t in tpls], visible=True, value="— pick a template —")
    for l in LORA_REGISTRY:
        if l["name"] in selection:
            lid = l["id"]
            d = LORA_DETAIL.get(lid, {})
            tpls = LORA_TEMPLATES.get(lid, [])
            is_portrait = lid in ("face_punch","amgery","sorpresa","snek","melt","cakeify","deflate","sprout","tear","unpaint","fat_elvis")
            best_mode = "I2V (upload portrait photo)" if lid in ("face_punch","amgery","sorpresa","snek") else "T2V or I2V" if is_portrait else "T2V"
            best_dur = "3-5s" if lid in ("face_punch","amgery","sorpresa","snek") else "5-8s" if l["cat"] in ("Effect","Transform") else "8-15s"
            info = f'### {l["name"]}\n\n'
            info += f'{d.get("ui", l["desc"])}\n\n'
            info += f'**Trigger:** `{l["trigger"]}` *(auto-injected by enhancer)*\n\n'
            info += f'**Best mode:** {best_mode} | **Best duration:** {best_dur} | **Category:** {l["cat"]}\n\n'
            info += f'**VRAM:** 805MB → ~3.8GB free → ~9.7s native at 480p\n\n'
            info += "**How to use:**\n"
            if lid in ("face_punch","amgery","sorpresa","snek"):
                info += "1. Upload a clear portrait photo (I2V mode)\n"
                info += "2. Write what effect you want (e.g. 'punch the face' or 'angry expression')\n"
                info += "3. Click Enhance — the LLM will format it properly with trigger words\n"
            elif l["cat"] == "Camera":
                info += "1. Describe your scene naturally (e.g. 'autumn forest from above')\n"
                info += "2. Click Enhance — the LLM adds the camera movement + trigger\n"
                info += "3. For I2V: describe only how camera should move around the subject\n"
            elif l["cat"] == "Style":
                info += "1. Describe any scene — the style applies to everything\n"
                info += "2. Click Enhance — the LLM adds style-specific visual language + trigger\n"
            else:
                info += "1. Describe the object/subject and the transformation\n"
                info += "2. Click Enhance — the LLM formats the effect description + trigger\n"
                info += "3. For I2V portraits: just describe the effect you want\n"
            return info, gr.update(choices=["— pick a template —"]+[t["title"] for t in tpls], visible=True, value="— pick a template —")
    return "", gr.update(visible=False)

def on_lora_template(tpl_name, lora_sel):
    if not tpl_name or tpl_name.startswith("—"): return gr.update(), gr.update()
    lid = "none"
    if lora_sel and not lora_sel.startswith("None"):
        for l in LORA_REGISTRY:
            if l["name"] in lora_sel: lid = l["id"]; break
    for t in LORA_TEMPLATES.get(lid, LORA_TEMPLATES.get("none",[])):
        if t["title"] == tpl_name:
            dur_opts = build_dur_options("480p")
            val = next((d for d in dur_opts if d.startswith(f'{t["dur"]}s')), dur_opts[min(3,len(dur_opts)-1)])
            return t["prompt"], val
    return gr.update(), gr.update()

# ── Core: Enhance ──

def on_enhance(prompt, neg, dur_label, q, lora_sel):
    if not prompt.strip(): return "⚠️ Enter a prompt first.", neg, get_stats()
    m = _re.search(r'(\d+)s', dur_label)
    sec = int(m.group(1)) if m else 8
    # Resolve LoRA
    lora_ids = []
    if lora_sel and not lora_sel.startswith("None"):
        for e in LORA_REGISTRY:
            if e["name"] in lora_sel: lora_ids.append(e["id"]); break
    is_i2v = False  # can't know here — but user prompt should indicate
    # Heuristic: if prompt mentions "subject", "person", "face" → likely I2V intent
    lower = prompt.lower()
    if any(w in lower for w in ["subject","person","face","portrait","photo","image","picture","uploaded","selfie","me ","my "]):
        is_i2v = True
    t0 = _time.time()
    enh_pos = enhance_single_prompt(prompt, sec, lora_ids=lora_ids if lora_ids else None, is_i2v=is_i2v)
    enh_neg = enhance_negative(neg, prompt)
    elapsed = _time.time() - t0
    wc = len(enh_pos.split()); tok = count_tokens(enh_pos)
    lora_tag = lora_ids[0] if lora_ids else "base"
    mode_tag = "I2V" if is_i2v else "T2V"
    return f"[{sec}s | {mode_tag} | {lora_tag} | {wc}w | {tok}tok | {elapsed:.1f}s]\n\n{enh_pos}", enh_neg, get_stats()

# ── Core: Generate ──

def on_gen(enh_pos, enh_neg, q, asp, dur_label, seed, img, lora_sel, lstr, progress=gr.Progress()):
    logs = []
    def lcb(m): logs.append(m)
    try:
        if not enh_pos or len(enh_pos.strip()) < 10:
            return None, "⚠️ Click 'Enhance' first!", "\n".join(logs), get_stats()

        m = _re.search(r'(\d+)s', dur_label)
        sec = int(m.group(1)) if m else 5
        nf = ltx_align(int(sec * GEN_FPS))
        w, h = resolve_dims(q, asp, img)
        sd = int(seed) if int(seed) != -1 else random.randint(0, 2**32-1)

        # Strip header
        clean = _re.sub(r'^\[.*?\]\n\n', '', enh_pos).strip()
        if not clean: clean = enh_pos

        # ── Load LoRA ──
        lora_id = None
        if lora_sel and not lora_sel.startswith("None"):
            for e in LORA_REGISTRY:
                if e["name"] in lora_sel: lora_id = e["id"]; break
        if lora_id:
            progress(0.02, "Loading LoRA...")
            sels = [(lora_id, LORA_BY_ID[lora_id].get("strength",0.8) * (lstr or 1.0))]
            ok, msg = load_loras(sels)
            lcb(msg)
            triggers = get_active_triggers()
            if triggers and triggers not in clean:
                clean = f"{triggers} {clean}"
        else:
            unload_loras()

        # ── Generate ──
        neg_final = enh_neg if enh_neg else DEFAULT_NEG
        fr, meta = generate_video(clean, neg_final, nf, w, h, sd,
            cond_img=img, progress_cb=lambda f,m: progress(f, desc=m), log_cb=lcb)

        # ── Cleanup ──
        unload_loras()
        progress(0.95, "Saving...")
        path = save_video(fr)
        lname = LORA_BY_ID[lora_id]["name"] if lora_id else "None"
        info = (f"{meta['width']}x{meta['height']} | {meta['frames']}f ({meta['duration_s']:.1f}s)\n"
                f"Time: {meta['gen_time_s']:.0f}s | Seed: {meta['seed']} | Chunks: {meta['chunks']}\n"
                f"LoRA: {lname}")
        return path, info, "\n".join(logs), get_stats()
    except Exception as e:
        traceback.print_exc()
        unload_loras()
        return None, f"Error: {e}", "\n".join(logs + [str(e)]), get_stats()

# ── Build UI ──
try: gr.close_all()
except: pass

init_dur = build_dur_options("480p")
_lora_choices = ["None (Base Model)"] + [f'{l["name"]} — {l["desc"]}' for l in LORA_REGISTRY]

with gr.Blocks(title="LTX-Video 13B + LoRA") as demo:
    gr.Markdown("# 🎬 LTX-Video 13B Pipeline\n15s native 480p · 30s+ chunked · 20 LoRA adapters · Free Kaggle T4×2")
    with gr.Row():
        stats = gr.Textbox(label="GPU", value=get_stats(), interactive=False, scale=3)
        gr.Button("Flush", size="sm", scale=1).click(
            lambda: (unload_loras(), gc.collect(), torch.cuda.empty_cache(), get_stats())[-1], outputs=stats)

    with gr.Row():
        # ── LEFT: Controls ──
        with gr.Column(scale=2):
            # LoRA Selection
            gr.Markdown("### 🎨 Style / Effect")
            s_lora = gr.Dropdown(choices=_lora_choices, value="None (Base Model)", label="LoRA Adapter",
                info="Select a LoRA or None for base model. One at a time (805MB VRAM).")
            s_lora_info = gr.Markdown("*Select a LoRA to see details and usage guide*")
            s_tpl = gr.Dropdown(choices=[], label="📝 Prompt Templates", visible=False)
            s_lstr = gr.Slider(0.3, 1.3, value=1.0, step=0.05, label="LoRA Strength",
                info="0.7=subtle  1.0=standard  >1.0=exaggerated", visible=True)

            gr.Markdown("---")
            gr.Markdown("### ✍️ Prompt")
            s_prompt = gr.Textbox(label="Your scene description", lines=3,
                placeholder="Describe naturally... e.g. 'autumn forest from above' or 'make face punch effect'")
            s_neg = gr.Textbox(label="Negative", lines=1, value=DEFAULT_NEG)

            gr.Markdown("---")
            gr.Markdown("### ⚙️ Settings")
            with gr.Row():
                s_q = gr.Dropdown(list(QUALITY_PRESETS.keys()), value="480p", label="Quality")
                s_asp = gr.Dropdown(list(ASPECT_MAP.keys()), value="16:9 Landscape", label="Aspect")
            s_dur = gr.Dropdown(init_dur, value=init_dur[min(3,len(init_dur)-1)], label="Duration")
            s_seed = gr.Number(label="Seed (-1=random)", value=-1, precision=0)
            s_img = gr.Image(label="📷 Input Image (for I2V mode)", type="pil")

            gr.Markdown("---")
            enhance_btn = gr.Button("① Enhance Prompt ✨", variant="secondary", size="lg")
            gen_btn = gr.Button("② Generate Video 🎬", variant="primary", size="lg")

        # ── RIGHT: Output ──
        with gr.Column(scale=3):
            gr.Markdown("### Enhanced Prompt *(review & edit before generating)*")
            s_enh = gr.Textbox(label="Enhanced Positive", lines=5, interactive=True,
                placeholder="Click 'Enhance Prompt' first — the LLM will rewrite your prompt for optimal results")
            s_neg_out = gr.Textbox(label="Enhanced Negative", lines=2, interactive=True)
            s_vid = gr.Video(label="Generated Video", height=400)
            s_info = gr.Textbox(label="Generation Info", lines=3, interactive=False)
            with gr.Accordion("📋 Generation Log", open=False):
                s_log = gr.Textbox(label="Log", lines=10, interactive=False)

    # ── Event Wiring ──
    s_lora.change(on_lora_change, s_lora, [s_lora_info, s_tpl])
    s_tpl.change(on_lora_template, [s_tpl, s_lora], [s_prompt, s_dur])
    s_q.change(lambda q: gr.update(choices=build_dur_options(q), value=build_dur_options(q)[min(3,len(build_dur_options(q))-1)]), s_q, s_dur)

    enhance_btn.click(on_enhance, [s_prompt, s_neg, s_dur, s_q, s_lora], [s_enh, s_neg_out, stats])
    gen_btn.click(on_gen, [s_enh, s_neg_out, s_q, s_asp, s_dur, s_seed, s_img, s_lora, s_lstr],
        [s_vid, s_info, s_log, stats])

print("UI built ✅")
demo.queue(max_size=2)
demo.launch(share=True, debug=False, show_error=True, inline=True)


UI built ✅
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://25c53165f76e9c1fd5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  NIM+: 55 words for 10s
  NIM neg: worst quality, inconsistent motion, blurry, jittery, distorted, morphing, flicke...
  🤖 Agent: 0 LoRA(s), 54w prompt
832x480 | 297f (9.9s) | Seed: 42 | Chunks: 1
timesteps=[1000, 993, 987, 981, 975, 909, 725] | tone_map=off
Chunk 1/1 (297f) T2V...


  0%|          | 0/7 [00:00<?, ?it/s]

    Decoding on cuda:1...
  -> 297 frames
Done: 297f (9.9s) in 343s
Saved: /kaggle/working/outputs/ltxv_20260406_175344_85a84904.mp4
  🤖 Agent: 0 LoRA(s), 55w prompt
832x480 | 297f (9.9s) | Seed: 42 | Chunks: 1
timesteps=[1000, 993, 987, 981, 975, 909, 725] | tone_map=off
Chunk 1/1 (297f) T2V...


  0%|          | 0/7 [00:00<?, ?it/s]

...

    Decoding on cuda:1...
  -> 153 frames
Done: 153f (5.1s) in 173s
Saved: /kaggle/working/outputs/ltxv_20260406_181413_3eacedce.mp4
  NIM+: 55 words for 8s
  NIM neg: worst quality, inconsistent motion, blurry, jittery, distorted, morphing, flicke...
  🤖 Agent: 1 LoRA(s), 36w prompt
  ✅ 1 LoRA(s): 🎬 Bullet Time (Σw=0.7) | 3.0GB free
832x480 | 241f (8.0s) | Seed: 42 | Chunks: 1
timesteps=[1000, 993, 987, 981, 975, 909, 725] | tone_map=off
Chunk 1/1 (241f) T2V...


  0%|          | 0/7 [00:00<?, ?it/s]

    Decoding on cuda:1...
  -> 241 frames
Done: 241f (8.0s) in 293s
Saved: /kaggle/working/outputs/ltxv_20260406_182051_a37c2486.mp4
  NIM+: 49 words for 8s
  NIM neg: worst quality, inconsistent motion, blurry, jittery, distorted, morphing, flicke...
  🤖 Agent: 1 LoRA(s), 46w prompt
  ✅ 1 LoRA(s): 🎬 Bullet Time (Σw=0.7) | 3.0GB free
832x480 | 241f (8.0s) | Seed: 42 | Chunks: 1
timesteps=[1000, 993, 987, 981, 975, 909, 725] | tone_map=off
Chunk 1/1 (241f) T2V...


  0%|          | 0/7 [00:00<?, ?it/s]

    Decoding on cuda:1...
  -> 241 frames
Done: 241f (8.0s) in 293s
Saved: /kaggle/working/outputs/ltxv_20260406_182750_53420d2a.mp4


## User Guide

### Workflow: Enhance → Generate
1. **Select LoRA** (or None) → info panel shows what it does + how to use it
2. **Pick a template** or write your own scene description naturally
3. **Click Enhance** → LLM rewrites your prompt with trigger words, camera specs, and optimal structure
4. **Review the enhanced prompt** (editable) → adjust if needed
5. **Click Generate** → video appears on the right

### For I2V (Image-to-Video)
- Upload a portrait/photo in the Image input
- Write what you want to HAPPEN (not what's in the image)
- Examples: "make the face melt", "punch effect", "slow camera orbit"
- The enhancer knows I2V mode and will NOT describe image contents

### LoRA Categories
| Category | Best For | Duration | Mode |
|----------|----------|----------|------|
| 🎬 Camera | Movement patterns | 8-15s | T2V |
| 🎨 Style | Visual aesthetics | 5-15s | T2V/I2V |
| ✨ Effect | Transformations | 3-8s | T2V/I2V |
| 🔄 Transform | Object changes | 5-10s | T2V/I2V |
| 😤 Expression | Face effects | 3-5s | I2V |

### IC-LoRAs (Not Available)
Depth, pose, canny, and detailer IC-LoRAs require a video-to-video pipeline.
Our notebook only supports T2V and I2V. These would need a second pipeline pass.

### VRAM & Duration
| Config | Free VRAM | Max native (480p) | Duration |
|--------|-----------|-------------------|----------|
| No LoRA | 4.6 GB | 449 frames | 15.0s |
| 1 LoRA | 3.8 GB | ~290 frames | 9.7s |
| With OOM retry + auto-extend | — | recovers gracefully | — |

### Token Limit
T5 max = 128 tokens ≈ 80 words. The enhancer targets 50-70 words. Longer prompts are truncated silently.
